In [ ]:
!pip install -q Sastrawi stopwordsiso gensim

import stopwordsiso as stopwords
stopwords_id = stopwords.stopwords('id')

# [v1_4_2] MO-WOA+ with two objectives

Changes from v1_4_1:
- optimizer (cell 20) is now MO-WOA+. whale update, objectives, repair, init, budget (30 + 50x29 = 1480 evals) and knee weights [32, 1] are still the same. added: external archive [A], (mu+lambda) elitist survival [E], per-whale guide from the archive [G], V-shaped transfer function [V], and 10 swap probes per generation [LS]
- `summarize()` now calls `mowoa_plus_algorithm_with_pareto(..., seed=0)`. also fixed the `init_objs[:,2]` crash (leftover from the 3-objective version) and the wrong axis in the pareto plot, and added the new components to the config table
- json output renamed to `mowoa_plus_summaries_150_seed0.json` so the v1_4_1 file doesn't get overwritten

checked against `run_mowoa_final` in the evaluation notebook: same knee subsets and same archive fronts (4 article sizes x 3 seeds).

# Download Indosum Dataset

In [ ]:
# download indosum dari kaggle
!pip install -q kagglehub

import kagglehub
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

base_path = kagglehub.dataset_download("linkgish/indosum")
path = f"{base_path}/indosum"
print("dataset path:", path)
!ls -R {path}

# pakai fold 01 dulu
temp_train_df = pd.read_json(f'{path}/train.01.jsonl', lines=True)
temp_test_df = pd.read_json(f'{path}/test.01.jsonl', lines=True)
temp_val_df = pd.read_json(f'{path}/dev.01.jsonl', lines=True)  # dev = validation

train_df = temp_train_df[['paragraphs', 'summary']].copy()
test_df = temp_test_df[['paragraphs', 'summary']].copy()
val_df = temp_val_df[['paragraphs', 'summary']].copy()

# paragraphs -> sentences -> words, digabung jadi satu string per artikel
def flatten_article(paragraphs):
    return ' '.join(' '.join(' '.join(words) for words in para) for para in paragraphs)

train_df['flattened_paragraphs'] = train_df['paragraphs'].apply(flatten_article)

sample_idx = 0

## Check paragraphs data


In [ ]:
print(len(train_df.paragraphs))

## Check dataset statistics

In [ ]:
# statistik dataset untuk 150 artikel sampel (test split)

import pandas as pd
import numpy as np
import glob

def load_all_splits(base_path):
    splits = {}
    for split_name, prefix in [('Train', 'train'), ('Validation', 'dev'), ('Test', 'test')]:
        files = sorted(glob.glob(f"{base_path}/{prefix}.*.jsonl"))
        if not files:
            continue
        dfs = [pd.read_json(f, lines=True)[['paragraphs', 'summary']] for f in files]
        splits[split_name] = pd.concat(dfs, ignore_index=True)
    return splits

print("Loading all dataset splits...")
splits = load_all_splits(path)

# ambil 150 artikel dari test split
SAMPLE_SIZE = 150
RANDOM_SEED = 42

eval_df = (splits['Test']
           .copy()
           .reset_index(drop=True)
           .sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
           .reset_index(drop=True))

print(f"{len(eval_df)} docs sampled from test split (seed {RANDOM_SEED})")

# fitur per artikel
def n_paragraphs(p):         return len(p)
def n_sentences_article(p):  return sum(len(para) for para in p)
def n_words_article(p):      return sum(len(s) for para in p for s in para)
def avg_words_per_sent(p):
    lens = [len(s) for para in p for s in para]
    return np.mean(lens) if lens else 0.0
def n_sentences_summary(s):  return len(s)
def n_words_summary(s):      return sum(len(sent) for sent in s)
def comp_ratio_sent(p, s):
    n = n_sentences_article(p)
    return n_sentences_summary(s) / n if n > 0 else 0.0
def comp_ratio_word(p, s):
    n = n_words_article(p)
    return n_words_summary(s) / n if n > 0 else 0.0
def build_vocab(df):
    vocab = set()
    for paragraphs in df['paragraphs']:
        for para in paragraphs:
            for sent in para:
                vocab.update(w.lower() for w in sent)
    return vocab

print("Computing features...")
feat = pd.DataFrame()
feat['n_paragraphs']        = eval_df['paragraphs'].apply(n_paragraphs)
feat['n_sentences_article'] = eval_df['paragraphs'].apply(n_sentences_article)
feat['n_words_article']     = eval_df['paragraphs'].apply(n_words_article)
feat['avg_words_per_sent']  = eval_df['paragraphs'].apply(avg_words_per_sent)
feat['n_sentences_summary'] = eval_df['summary'].apply(n_sentences_summary)
feat['n_words_summary']     = eval_df['summary'].apply(n_words_summary)
feat['comp_ratio_sent']     = eval_df.apply(
    lambda r: comp_ratio_sent(r['paragraphs'], r['summary']), axis=1)
feat['comp_ratio_word']     = eval_df.apply(
    lambda r: comp_ratio_word(r['paragraphs'], r['summary']), axis=1)

vocab_size = len(build_vocab(eval_df))

# tabel ringkasan
table = pd.DataFrame([
    ('Total Documents',                    f"{len(eval_df):,}"),
    ('Source Split',                       'Test'),
    ('Random Seed',                        f"{RANDOM_SEED}"),
    ('Vocabulary Size (Unique Tokens)',     f"{vocab_size:,}"),
    ('Avg Paragraphs per Article',         f"{feat['n_paragraphs'].mean():.2f}"),
    ('Avg Sentences per Article',          f"{feat['n_sentences_article'].mean():.2f}"),
    ('Min Sentences per Article',          f"{int(feat['n_sentences_article'].min())}"),
    ('Max Sentences per Article',          f"{int(feat['n_sentences_article'].max())}"),
    ('Avg Words per Article',              f"{feat['n_words_article'].mean():.2f}"),
    ('Avg Words per Sentence (Article)',   f"{feat['avg_words_per_sent'].mean():.2f}"),
    ('Avg Sentences per Summary',          f"{feat['n_sentences_summary'].mean():.2f}"),
    ('Min Sentences per Summary',          f"{int(feat['n_sentences_summary'].min())}"),
    ('Max Sentences per Summary',          f"{int(feat['n_sentences_summary'].max())}"),
    ('Avg Words per Summary',              f"{feat['n_words_summary'].mean():.2f}"),
    ('Avg Compression Ratio (Sentences)',  f"{feat['comp_ratio_sent'].mean():.3f}"),
    ('Avg Compression Ratio (Words)',      f"{feat['comp_ratio_word'].mean():.3f}"),
], columns=['Metric', 'Value'])

table = table.set_index('Metric')

print(f"\nIndoSum eval subset, {len(eval_df)} docs (test split, seed {RANDOM_SEED})\n")
print(table.to_string())

# Initialize Pre-processing Functions

In [ ]:
import numpy as np
import networkx as nx
import Sastrawi
import re

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

def flatten_and_join(nested):
    result = []
    for item in nested:
        if isinstance(item, list):
            result.append(flatten_and_join(item))
        else:
            result.append(item)
    return ' '.join(result)

In [ ]:
_factory = StemmerFactory()
_stemmer = _factory.create_stemmer()

def preprocess_sentence(sentence):
    indonesian_stopwords = stopwords_id

    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z\s]', '', sentence)  # huruf & spasi saja
    words = sentence.split()
    words = [w for w in words if w not in indonesian_stopwords]
    # stemmer dibuat sekali di atas, sastrawi lambat kalau dibuat ulang tiap kalimat
    words = [_stemmer.stem(w) for w in words]
    return words

In [ ]:
def split_text_into_sentences(paragraphs):
    # indosum sudah ditokenisasi: paragraphs -> sentences -> words
    # jadi cukup join per kalimat, tidak perlu sentence splitter
    sentences = []
    for paragraph in paragraphs:
        for sentence_tokens in paragraph:
            sentence = ' '.join(sentence_tokens)

            # rapikan spasi sebelum tanda baca
            sentence = sentence.replace(' ,', ',')
            sentence = sentence.replace(' .', '.')
            sentence = sentence.replace(' :', ':')
            sentence = sentence.replace(' ;', ';')
            sentence = sentence.replace(' !', '!')
            sentence = sentence.replace(' ?', '?')
            sentence = sentence.replace(' )', ')')
            sentence = sentence.replace('( ', '(')

            if sentence.strip():
                sentences.append(sentence.strip())
    return sentences

In [ ]:
def preprocess_paragraph(paragraphs):
  # versi lama, sekarang pakai split_text_into_sentences
  document_text = flatten_and_join(paragraphs)
  print(document_text)

  raw_sentences = re.split(r'(?<=[.?!])\s+', document_text.strip())
  sentences = [s.strip() for s in raw_sentences if s.strip()]
  print(f"Detected {len(sentences)} sentences.")
  print("sentences: ", sentences)

  factory = StemmerFactory()
  stemmer = factory.create_stemmer()

  indonesian_stopwords = stopwords_id

  return [preprocess_sentence(s) for s in sentences]

# Import BGE-M3 Model

In [ ]:
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer

# bge-m3 multilingual, bisa langsung pakai kalimat mentah (tanpa tokenisasi)
print("Loading BGE-M3 model...")
model = SentenceTransformer('BAAI/bge-m3')
print("BGE-M3 loaded")

## Invoke model and output graph function

In [ ]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 11

from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate
from mpl_toolkits.mplot3d import Axes3D

# report_mode = True -> print tabel & simpan gambar untuk laporan
report_mode = True
REPORT_FIG_DIR = "report_figures"

if report_mode:
    os.makedirs(REPORT_FIG_DIR, exist_ok=True)


def _report_header(title):
    print(f"\n===== {title} =====")

In [ ]:
def run_bge_m3_and_output_graph(sentences, article_id=None):
    """embedding bge-m3 -> cosine similarity -> graph kalimat. return (G, report_data)"""
    label = f"Article {article_id}" if article_id is not None else "Article"

    # embedding sudah dinormalisasi
    sentence_vectors = model.encode(sentences, normalize_embeddings=True)

    cosine_sim = cosine_similarity(sentence_vectors)

    # edge hanya kalau similarity > threshold
    threshold = 0.1
    G = nx.Graph()
    for i, s in enumerate(sentences):
        G.add_node(i, text=s.strip(), original_text=s.strip())

    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if cosine_sim[i, j] > threshold:
                G.add_edge(i, j, weight=cosine_sim[i, j])

    # data untuk laporan
    n = len(sentences)
    max_possible_edges = n * (n - 1) // 2
    edge_weights = [d['weight'] for _, _, d in G.edges(data=True)]

    report_data = {
        'sentence_vectors': sentence_vectors,
        'cosine_sim': cosine_sim,
        'threshold': threshold,
        'num_nodes': G.number_of_nodes(),
        'num_edges': G.number_of_edges(),
        'max_possible_edges': max_possible_edges,
        'graph_density': G.number_of_edges() / max_possible_edges if max_possible_edges > 0 else 0,
        'edge_weights': np.array(edge_weights),
        'embedding_dim': sentence_vectors.shape[1],
    }

    if report_mode:
        _report_header(f"Embedding & graph - {label}")

        # ringkasan embedding
        print(f"bge-m3, {n} kalimat, dim {sentence_vectors.shape[1]}, "
              f"mean L2 norm {np.linalg.norm(sentence_vectors, axis=1).mean():.4f}")

        # contoh 5 dimensi pertama tiap kalimat
        print("\n5 dimensi pertama tiap embedding:")
        sample_rows = []
        for i in range(n):
            vec_preview = ", ".join(f"{v:+.4f}" for v in sentence_vectors[i, :5])
            sent_preview = sentences[i].strip()[:60] + ("..." if len(sentences[i].strip()) > 60 else "")
            sample_rows.append([i, sent_preview, f"[{vec_preview}, ...]"])
        print(tabulate(sample_rows, headers=["Idx", "Sentence (truncated)", "Embedding [d0..d4, ...]"],
                       tablefmt="grid", maxcolwidths=[5, 60, 50]))

        # matrix lengkap, tab-separated biar gampang dipaste ke word
        print(f"\ncosine similarity {n}x{n} (tab-separated):")
        print("\t".join([""] + [f"S{j}" for j in range(n)]))
        for i in range(n):
            row_vals = "\t".join(f"{cosine_sim[i, j]:.4f}" for j in range(n))
            print(f"S{i}\t{row_vals}")

        # simpan juga sebagai tsv
        tsv_path = os.path.join(REPORT_FIG_DIR, f"cosine_similarity_matrix_{article_id}.tsv")
        with open(tsv_path, "w", encoding="utf-8") as f:
            f.write("\t".join([""] + [f"S{j}" for j in range(n)]) + "\n")
            for i in range(n):
                row_vals = "\t".join(f"{cosine_sim[i, j]:.4f}" for j in range(n))
                f.write(f"S{i}\t{row_vals}\n")
        print("saved", tsv_path)

        # heatmap
        fig, ax = plt.subplots(figsize=(8, 6.5))
        im = ax.imshow(cosine_sim, cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_xlabel("Sentence Index")
        ax.set_ylabel("Sentence Index")
        ax.set_title(f"Cosine Similarity Heatmap - {label}")
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        plt.colorbar(im, ax=ax, label="Cosine Similarity")
        plt.tight_layout()
        heatmap_path = os.path.join(REPORT_FIG_DIR, f"similarity_heatmap_{article_id}.png")
        plt.savefig(heatmap_path, dpi=200)
        plt.show()
        print("saved", heatmap_path)

        # info graph
        print("\ngraph:")
        graph_table = [
            ["Number of nodes (sentences)", report_data['num_nodes']],
            ["Number of edges", report_data['num_edges']],
            ["Maximum possible edges", report_data['max_possible_edges']],
            ["Graph density", f"{report_data['graph_density']:.4f}"],
            ["Similarity threshold (τ)", report_data['threshold']],
        ]
        print(tabulate(graph_table, headers=["Property", "Value"], tablefmt="grid"))

        # distribusi bobot edge
        if len(edge_weights) > 0:
            ew = report_data['edge_weights']
            print("\nbobot edge:")
            ew_stats = [
                ["Count", len(ew)],
                ["Min", f"{ew.min():.4f}"],
                ["Max", f"{ew.max():.4f}"],
                ["Mean", f"{ew.mean():.4f}"],
                ["Median", f"{np.median(ew):.4f}"],
                ["Std Dev", f"{ew.std():.4f}"],
            ]
            print(tabulate(ew_stats, headers=["Statistic", "Value"], tablefmt="grid"))

            fig, ax = plt.subplots(figsize=(7, 4))
            ax.hist(ew, bins=20, edgecolor='black', color='steelblue', alpha=0.85)
            ax.axvline(ew.mean(), color='red', linestyle='--', label=f'Mean = {ew.mean():.3f}')
            ax.set_xlabel("Cosine Similarity (Edge Weight)")
            ax.set_ylabel("Frequency")
            ax.set_title(f"Edge Weight Distribution - {label}")
            ax.legend()
            plt.tight_layout()
            hist_path = os.path.join(REPORT_FIG_DIR, f"edge_weight_dist_{article_id}.png")
            plt.savefig(hist_path, dpi=200)
            plt.show()
            print("saved", hist_path)

        # visualisasi graph
        fig, ax = plt.subplots(figsize=(8, 6))
        pos = nx.spring_layout(G, seed=42, k=1.5/np.sqrt(n))
        edge_weights_draw = [G[u][v]['weight'] for u, v in G.edges()]
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=500, node_color='skyblue',
                               edgecolors='black', linewidths=0.8)
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_weight='bold')
        nx.draw_networkx_edges(G, pos, ax=ax, width=[w * 2 for w in edge_weights_draw],
                               alpha=0.6, edge_color='gray')
        ax.set_title(f"Sentence Similarity Graph - {label}\n"
                     f"(Nodes={G.number_of_nodes()}, Edges={G.number_of_edges()}, τ={threshold})")
        ax.axis('off')
        plt.tight_layout()
        graph_path = os.path.join(REPORT_FIG_DIR, f"similarity_graph_{article_id}.png")
        plt.savefig(graph_path, dpi=200, bbox_inches='tight')
        plt.show()
        print("saved", graph_path)

        # cek sensitivitas threshold (jumlah edge / konektivitas)
        print("\nsensitivitas threshold:")
        thresholds_to_test = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
        sensitivity_rows = []
        for t in thresholds_to_test:
            edge_count = sum(1 for i in range(n) for j in range(i+1, n) if cosine_sim[i, j] > t)
            density = edge_count / max_possible_edges if max_possible_edges > 0 else 0
            G_temp = nx.Graph()
            G_temp.add_nodes_from(range(n))
            for i in range(n):
                for j in range(i+1, n):
                    if cosine_sim[i, j] > t:
                        G_temp.add_edge(i, j)
            connected = nx.is_connected(G_temp)
            num_components = nx.number_connected_components(G_temp)
            marker = " ← selected" if t == threshold else ""
            sensitivity_rows.append([t, edge_count, f"{density:.4f}", num_components,
                                     "Yes" if connected else "No", marker])
        print(tabulate(sensitivity_rows,
                       headers=["Threshold (τ)", "Edges", "Density", "Components", "Connected?", ""],
                       tablefmt="grid"))

    return G, report_data

# Sentence importance with LexRank function

In [ ]:
def compute_lexrank_importance(graph, damping=0.85):
    # # Ensure graph is undirected
    # if not isinstance(graph, nx.Graph):
    #     graph = graph.to_undirected()

    # LexRank is basically weighted PageRank
    scores = nx.pagerank(graph, weight='weight', alpha=damping)
    print(scores)

    nx.set_node_attributes(graph, scores, 'importance')
    return scores

# MOWOA Function

In [ ]:
# MO-WOA+ (pengganti cell 20 di v1_4_1)
# tambahan dari MO-WOA biasa:
#   [A]  external archive, max 30, dipangkas pakai crowding distance
#   [E]  survival (mu+lambda): non-dominated sorting + crowding
#   [G]  guide tiap whale diambil dari archive (tournament crowding distance, spt MOPSO-CD)
#   [V]  transfer function V-shaped |tanh(step)|, gantinya sigmoid
#   [LS] 10 swap probe per generasi ke anggota archive (swap -> jumlah kalimat tetap, repair tidak jalan)
# sisanya sama: persamaan whale, a 2->0, split 70/30, slot leader dipertahankan
# (30 + 50*29 = 1480 evaluasi), repair, inisialisasi, knee weights [32, 1]
# seed=0 -> hasil sama dengan run fase C

import numpy as np
import networkx as nx


def is_dominated(obj1, obj2):
    return (obj2[0] >= obj1[0] and
            obj2[1] <= obj1[1] and
            (obj2[0] > obj1[0] or obj2[1] < obj1[1]))


def compute_pareto_front(population, objectives):
    """index + objektif dari solusi yang non-dominated"""
    pareto_indices = []
    pareto_front = []
    for i, obj1 in enumerate(objectives):
        dominated = False
        for j, obj2 in enumerate(objectives):
            if i != j and is_dominated(obj1, obj2):
                dominated = True
                break
        if not dominated:
            pareto_indices.append(i)
            pareto_front.append(tuple(obj1))
    return pareto_indices, pareto_front


def crowding_distance(objectives):
    """crowding distance NSGA-II, titik ujung = inf"""
    n_solutions = len(objectives)
    n_objectives = objectives.shape[1]
    if n_solutions <= 2:
        return np.full(n_solutions, np.inf)
    distances = np.zeros(n_solutions)
    for obj_idx in range(n_objectives):
        sorted_indices = np.argsort(objectives[:, obj_idx])
        distances[sorted_indices[0]] = np.inf
        distances[sorted_indices[-1]] = np.inf
        obj_range = (objectives[sorted_indices[-1], obj_idx] -
                     objectives[sorted_indices[0], obj_idx])
        if obj_range == 0:
            continue
        for i in range(1, n_solutions - 1):
            distances[sorted_indices[i]] += \
                (objectives[sorted_indices[i + 1], obj_idx] -
                 objectives[sorted_indices[i - 1], obj_idx]) / obj_range
    return distances


def mowoa_plus_algorithm_with_pareto(graph, pop_size=30, max_iter=50,
                                     summary_size=0.25, position_bonus=None,
                                     seed=0, archive_size=30, ls_probes=10):
    num_sentences = len(graph.nodes())
    if summary_size >= 1:
        target_length = max(1, min(int(summary_size), num_sentences))
    else:
        target_length = max(1, int(num_sentences * summary_size))
    node_importance = np.array([graph.nodes[n].get('importance', 0)
                                for n in graph.nodes()])
    if np.all(node_importance == 0):
        node_importance = np.array([graph.degree(n, weight='weight')
                                    for n in graph.nodes()])
    adj_matrix = nx.to_numpy_array(graph, weight='weight')

    if position_bonus is None:
        position_bonus = np.ones(num_sentences)

    biased_importance = np.array([
        graph.nodes[i]['importance'] * position_bonus[i]
        for i in range(num_sentences)
    ])

    # 2 rng terpisah: satu untuk inisialisasi, satu untuk loop
    rng_init = np.random.default_rng(seed)
    rng = np.random.default_rng(seed)

    # objektif: coverage (max), redundancy (min)
    def evaluate_objectives(solution):
        selected = np.where(solution == 1)[0]
        coverage = biased_importance[selected].sum() if len(selected) > 0 else 0
        redundancy_raw = 0
        num_pairs = 0
        for i_idx, i in enumerate(selected):
            for j_idx, j in enumerate(selected):
                if j_idx > i_idx:
                    redundancy_raw += adj_matrix[i][j]
                    num_pairs += 1
        redundancy = redundancy_raw / max(1, num_pairs) if num_pairs > 0 else 0
        redundancy = max(0, redundancy - 0.2)
        return coverage, redundancy

    def fitness(solution):
        coverage, redundancy = evaluate_objectives(solution)
        return coverage - redundancy

    # repair: paksa jumlah kalimat = target_length
    def repair_solution(solution):
        selected = np.where(solution == 1)[0]
        num_selected = len(selected)
        if num_selected > target_length:
            drop_order = selected[np.argsort(biased_importance[selected])]
            for idx in drop_order[:num_selected - target_length]:
                solution[idx] = 0
        elif num_selected < target_length:
            unselected = np.where(solution == 0)[0]
            if len(unselected) > 0:
                add_order = unselected[np.argsort(-biased_importance[unselected])]
                for idx in add_order[:target_length - num_selected]:
                    solution[idx] = 1
        if not solution.any():
            solution[np.argmax(biased_importance)] = 1
        return solution

    # inisialisasi, peluang ~ importance * position bonus
    def initialize_population():
        init_scores = node_importance * position_bonus
        probs = init_scores / init_scores.sum() if init_scores.sum() > 0 else \
                np.ones_like(init_scores) / len(init_scores)
        population = []
        for _ in range(pop_size):
            solution = np.zeros(num_sentences, dtype=int)
            num_to_select = max(1, rng_init.integers(
                max(1, target_length - 1), target_length + 2))
            selected = rng_init.choice(num_sentences,
                                       size=min(int(num_to_select), num_sentences),
                                       replace=False, p=probs)
            solution[selected] = 1
            solution = repair_solution(solution)
            population.append(solution)
        return np.array(population)

    # [V] step -> peluang flip bit. a mengecil -> step mengecil -> child mirip parent
    def v_transfer(step):
        return np.abs(np.tanh(step))

    def make_offspring(current, guide, a, pop):
        p = rng.random()
        b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()
        new_pos = current.copy()
        if p < 0.5:
            ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
            for i in range(num_sentences):
                D = abs(C_coeff * ref[i] - current[i])
                if rng.random() < v_transfer(A_coeff * D):          # encircling
                    new_pos[i] = 1 - new_pos[i]                    # atau random search
        else:
            l = rng.uniform(-1, 1)
            for i in range(num_sentences):
                D_prime = abs(guide[i] - current[i])
                step = D_prime * np.exp(b * l) * np.cos(2 * np.pi * l)   # spiral
                if rng.random() < v_transfer(step):
                    new_pos[i] = 1 - new_pos[i]
        if not new_pos.any():
            new_pos[rng.integers(num_sentences)] = 1
        return repair_solution(new_pos)

    # [A] archive: non-dominated, tanpa duplikat, dipangkas kalau penuh
    def nondominated_mask_min(F):
        keep = np.ones(len(F), dtype=bool)
        for i in range(len(F)):
            if not keep[i]:
                continue
            le = np.all(F <= F[i], axis=1)
            lt = np.any(F < F[i], axis=1)
            if np.any(le & lt):
                keep[i] = False
        return keep

    def archive_update(B, F_min, arch_B, arch_F):
        allB = np.vstack([arch_B, B]) if len(arch_B) else B.copy()
        allF = np.vstack([arch_F, F_min]) if len(arch_F) else F_min.copy()
        keep = nondominated_mask_min(allF)
        allB, allF = allB[keep], allF[keep]
        _, uidx = np.unique(np.round(allF, 12), axis=0, return_index=True)
        allB, allF = allB[np.sort(uidx)], allF[np.sort(uidx)]
        if len(allF) > archive_size:
            cd = crowding_distance(allF)
            keep_idx = np.argsort(-cd)[:archive_size]
            allB, allF = allB[keep_idx], allF[keep_idx]
        return allB, allF

    # [E] seleksi (mu+lambda)
    def nds_crowding_select(B_all, F_all, k):
        remaining = np.arange(len(F_all))
        chosen = []
        while len(chosen) < k and len(remaining) > 0:
            mask = nondominated_mask_min(F_all[remaining])
            front = remaining[mask]
            if len(chosen) + len(front) <= k:
                chosen.extend(front.tolist())
                remaining = remaining[~mask]
            else:
                cd = crowding_distance(F_all[front])
                chosen.extend(front[np.argsort(-cd)[:k - len(chosen)]].tolist())
                break
        chosen = np.array(chosen[:k])
        return B_all[chosen], F_all[chosen]

    # [LS] swap 1 kalimat redundan dengan kalimat yang kurang mirip
    # jumlah kalimat tidak berubah jadi repair tidak ikut menarik ke kalimat penting
    def ls_probe(arch_B, arch_F):
        if len(arch_B) == 0:
            return None
        if len(arch_B) == 1:
            base = arch_B[0]
        else:
            cdp = crowding_distance(arch_F)
            a1, a2 = rng.integers(len(arch_B), size=2)
            base = arch_B[a1] if cdp[a1] >= cdp[a2] else arch_B[a2]
        x = np.asarray(base).astype(int).copy()
        sel = np.where(x == 1)[0]
        unsel = np.where(x == 0)[0]
        if len(sel) == 0 or len(unsel) == 0:
            return None
        contrib = np.array([adj_matrix[i, sel].sum() for i in sel])
        out_i = sel[int(np.argmax(contrib))] if rng.random() < 0.7 else rng.choice(sel)
        remaining = sel[sel != out_i]
        scores = adj_matrix[unsel][:, remaining].sum(axis=1)
        k = min(3, len(unsel))
        in_i = unsel[rng.choice(np.argsort(scores)[:k])]
        x[out_i] = 0
        x[in_i] = 1
        return x

    # [G] guide per whale dari archive (tournament crowding distance)
    def draw_guide(arch_B, arch_F, population, pareto_indices):
        if len(arch_B) > 0:
            if len(arch_B) == 1:
                return arch_B[0]
            cd = crowding_distance(arch_F)
            a1, a2 = rng.integers(len(arch_B), size=2)
            return arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]
        if len(pareto_indices) == 1:
            return population[pareto_indices[0]]
        cd = crowding_distance(np.array(
            [evaluate_objectives(population[i]) for i in pareto_indices]))
        a1, a2 = rng.integers(len(pareto_indices), size=2)
        return population[pareto_indices[a1 if cd[a1] >= cd[a2] else a2]]

    # main loop
    convergence_history = []
    pareto_size_history = []          # ukuran archive per iterasi
    leader_history = []               # wakil archive per iterasi
    n_evaluations = 0

    population = initialize_population()
    n_evaluations += pop_size

    # diagnostik populasi awal (untuk laporan)
    init_population_snapshot = population.copy()
    init_objectives_snapshot = np.array(
        [evaluate_objectives(sol) for sol in init_population_snapshot])
    init_selection_frequency = init_population_snapshot.sum(axis=0) / pop_size
    if pop_size >= 2:
        _diffs = (init_population_snapshot[:, None, :]
                  != init_population_snapshot[None, :, :])
        _iu = np.triu_indices(pop_size, k=1)
        init_hamming_diversity = float(
            _diffs.sum(axis=-1).astype(float)[_iu].mean() / num_sentences)
    else:
        init_hamming_diversity = 0.0
    if np.std(biased_importance) > 0 and np.std(init_selection_frequency) > 0:
        init_freq_importance_corr = float(
            np.corrcoef(biased_importance, init_selection_frequency)[0, 1])
    else:
        init_freq_importance_corr = 0.0

    F_native = init_objectives_snapshot.copy()
    F_min = np.column_stack([-F_native[:, 0], F_native[:, 1]])
    arch_B = np.empty((0, num_sentences), dtype=int)
    arch_F = np.empty((0, 2))
    arch_B, arch_F = archive_update(population, F_min, arch_B, arch_F)

    for iteration in range(max_iter):
        a = 2.0 - iteration * (2.0 / max_iter)

        F_native = np.array([evaluate_objectives(sol) for sol in population])
        F_min = np.column_stack([-F_native[:, 0], F_native[:, 1]])
        pareto_indices, _ = compute_pareto_front(population, F_native)
        if len(pareto_indices) == 0:
            pareto_indices = list(range(len(population)))
        leader_idx = 0                       # slot 0 tidak diubah -> 29 child

        # slot yang dapat LS probe generasi ini
        if ls_probes > 0 and len(arch_B) > 0:
            nonleader = [i for i in range(len(population)) if i != leader_idx]
            probe_slots = set(rng.choice(nonleader,
                                         size=min(ls_probes, len(nonleader)),
                                         replace=False).tolist())
        else:
            probe_slots = set()

        # buat 29 kandidat baru
        new_pop = []
        children = []
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)
                continue
            child = None
            if i in probe_slots:
                child = ls_probe(arch_B, arch_F)
            if child is None:
                if rng.random() < 0.7:
                    guide = draw_guide(arch_B, arch_F, population, pareto_indices)
                else:
                    guide = population[rng.choice(pareto_indices)]
                child = make_offspring(sol, guide, a, population)
            new_pop.append(child)
            children.append(child)
        children = np.array(children)
        n_evaluations += len(children)
        Fc_native = np.array([evaluate_objectives(c) for c in children])
        Fc_min = np.column_stack([-Fc_native[:, 0], Fc_native[:, 1]])

        # [E]
        B_all = np.vstack([population, children])
        F_all = np.vstack([F_min, Fc_min])
        population, _ = nds_crowding_select(B_all, F_all, pop_size)

        # [A]
        arch_B, arch_F = archive_update(children, Fc_min, arch_B, arch_F)

        # monitoring saja, tidak pakai rng
        fitnesses = np.array([fitness(sol) for sol in population])
        convergence_history.append(fitnesses.max())
        pareto_size_history.append(len(arch_F))
        if len(arch_F) > 1:
            cd_arch = crowding_distance(arch_F)
            rep = int(np.argmax(cd_arch))
            rep_cd = cd_arch[rep]
        else:
            rep, rep_cd = 0, np.inf
        rep_native = (-arch_F[rep, 0], arch_F[rep, 1])
        # range archive + knee per iterasi, untuk plot
        cov_all = -arch_F[:, 0]
        red_all = arch_F[:, 1]
        _nat = np.column_stack([cov_all, red_all])
        _omin, _omax = _nat.min(axis=0), _nat.max(axis=0)
        _rng = _omax - _omin
        _rng[_rng == 0] = 1
        _nrm = (_nat - _omin) / _rng
        _nrm[:, 0] = 1 - _nrm[:, 0]
        _kd = np.sqrt((np.array([32.0, 1.0]) * _nrm ** 2).sum(axis=1))
        _kn = int(_kd.argmin())
        leader_history.append({
            'iter': iteration,
            'leader_idx': rep,                      # index di archive
            'coverage': float(rep_native[0]),
            'redundancy': float(rep_native[1]),
            'crowding_distance': float(rep_cd),
            'is_boundary': bool(np.isinf(rep_cd)),
            'pareto_size': len(arch_F),
            'cov_min': float(cov_all.min()), 'cov_max': float(cov_all.max()),
            'red_min': float(red_all.min()), 'red_max': float(red_all.max()),
            'knee_cov': float(cov_all[_kn]), 'knee_red': float(red_all[_kn]),
            'arch_front': np.column_stack([cov_all, red_all]).copy(),
        })

    # front akhir = archive, pilih knee point
    archive_native = np.column_stack([-arch_F[:, 0], arch_F[:, 1]])
    final_pareto_front = [tuple(o) for o in archive_native]
    final_pareto_indices = list(range(len(arch_B)))

    if len(arch_B) > 0:
        pareto_objs_arr = archive_native
        obj_min = pareto_objs_arr.min(axis=0)
        obj_max = pareto_objs_arr.max(axis=0)
        obj_range = obj_max - obj_min
        obj_range[obj_range == 0] = 1
        normalized = (pareto_objs_arr - obj_min) / obj_range
        normalized[:, 0] = 1 - normalized[:, 0]     # coverage dibalik (max -> min)
        knee_weights = np.array([32.0, 1.0])        # [coverage, redundancy]
        distances = np.sqrt((knee_weights * normalized ** 2).sum(axis=1))
        knee_idx_in_pareto = int(distances.argmin())
        best_idx = knee_idx_in_pareto               # index di archive
        best_solution = arch_B[best_idx]
    else:                                           # harusnya tidak pernah terjadi
        fitnesses = np.array([fitness(sol) for sol in population])
        best_idx = int(fitnesses.argmax())
        best_solution = population[best_idx]
        distances = normalized = pareto_objs_arr = None
        knee_idx_in_pareto = None

    final_objs = np.array([evaluate_objectives(sol) for sol in population])

    mowoa_report = {
        'num_sentences': num_sentences,
        'target_length': target_length,
        'pop_size': pop_size,
        'max_iter': max_iter,
        'summary_size': summary_size,
        'convergence_history': np.array(convergence_history),
        'pareto_size_history': np.array(pareto_size_history),
        'final_front': final_pareto_front,          # isinya archive
        'final_objs': final_objs,
        'pareto_indices': final_pareto_indices,
        'pareto_objs': pareto_objs_arr,
        'pareto_normalized': normalized,
        'pareto_distances': distances,
        'knee_idx_in_pareto': knee_idx_in_pareto,
        'best_idx': best_idx,                       # index di archive
        'best_solution': best_solution,
        'best_objectives': evaluate_objectives(best_solution),
        'biased_importance': biased_importance,
        'position_bonus': position_bonus,
        'node_importance': node_importance,
        'init_population': init_population_snapshot,
        'init_objectives': init_objectives_snapshot,
        'init_selection_frequency': init_selection_frequency,
        'init_hamming_diversity': init_hamming_diversity,
        'init_freq_importance_corr': init_freq_importance_corr,
        'leader_history': leader_history,
        # tambahan MO-WOA+
        'seed': seed,
        'n_evaluations': n_evaluations,             # 30 + 50*29 = 1480
        'archive_size': archive_size,
        'ls_probes': ls_probes,
        'transfer': 'V-shaped |tanh(step)|',
        'survival': '(mu+lambda) NDS + crowding',
        'guide_mode': 'per-whale archive CD-tournament',
        'archive_solutions': arch_B,
        'archive_objectives': archive_native,
        'adj_matrix': adj_matrix,
    }

    threshold = summary_size
    recommended = []
    for sol in arch_B:
        cov, red = evaluate_objectives(sol)
        selected = np.where(sol == 1)[0]
        total_importance = node_importance[selected].sum() if len(selected) > 0 else 0
        if total_importance > threshold:
            recommended.append((selected, total_importance, (cov, red)))

    return best_solution, final_pareto_front, population, evaluate_objectives, \
           recommended, mowoa_report


# Evalute ROUGE Score Function

In [ ]:
!pip install rouge-score

from rouge_score import rouge_scorer

In [ ]:
def evaluate_rouge_score(best_solution, graph, golden_summary):
    # use_stemmer=False, stemmer bawaan Porter (inggris), tidak cocok untuk bahasa indonesia
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    generated_summary_parts = []
    for s_idx in np.where(best_solution == 1)[0]:
        generated_summary_parts.append(graph.nodes[s_idx]['text'])
    generated_summary = ' '.join(generated_summary_parts)

    print("generated summary: ", generated_summary)

    scores = scorer.score(golden_summary, generated_summary)
    return scores

In [ ]:
def lead_n_baseline(paragraphs, golden_summary, n=3):
    """baseline LEAD-N: ambil n kalimat pertama"""
    sentences = split_text_into_sentences(paragraphs)
    summary_sentences = sentences[:min(n, len(sentences))]
    generated_summary = ' '.join(summary_sentences)

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = scorer.score(golden_summary, generated_summary)
    return scores, generated_summary

# Core summarizer function

In [ ]:
def tokens_to_narrative(tokenized_sentences):
    """list kalimat (token) -> satu string"""
    sentences = []

    for tokens in tokenized_sentences:
        sentence = " ".join(tokens)

        # rapikan spasi tanda baca
        sentence = sentence.replace(" ,", ",")
        sentence = sentence.replace(" .", ".")
        sentence = sentence.replace(" :", ":")
        sentence = sentence.replace(" ;", ";")
        sentence = sentence.replace(" !", "!")
        sentence = sentence.replace(" ?", "?")

        sentences.append(sentence)

    return " ".join(sentences)


In [ ]:
debug = True

def dprint(*args, **kwargs):
    if debug:
        print(*args, **kwargs)

In [ ]:
def summarize(paragraphs, golden_summary, article_id=None):
    """pipeline lengkap: kalimat -> graph -> lexrank -> MO-WOA+ -> rouge.
    kalau report_mode True, semua tabel/gambar untuk laporan ikut dibuat."""
    # preprocessed_sentences = preprocess_paragraph(paragraphs)
    sentences = split_text_into_sentences(paragraphs)

    graph, embedding_report = run_bge_m3_and_output_graph(sentences, article_id=article_id)

    lexrank_scores = compute_lexrank_importance(graph)

    # bobot posisi, kalimat awal berita biasanya paling penting
    n = len(sentences)
    position_bonus = np.ones(n)
    position_bonus[0] = 3.0
    position_bonus[1:3] *= 2.5
    position_bonus[3:5] *= 1.5
    position_bonus[5:8] *= 0.8
    position_bonus[8:] *= 0.3

    # laporan: skor lexrank
    if report_mode:
        label = f"Article {article_id}" if article_id is not None else "Article"
        _report_header(f"LexRank - {label}")

        print("config lexrank:")
        config_table = [
            ["Algorithm", "Weighted PageRank (nx.pagerank)"],
            ["Damping factor (α)", "0.85"],
            ["Edge weights", "Cosine similarity (from BGE-M3 embeddings)"],
            ["Convergence", "Default NetworkX tolerance (1e-6)"],
        ]
        print(tabulate(config_table, headers=["Parameter", "Value"], tablefmt="grid"))

        # urut dari skor tertinggi
        importance_arr = np.array([graph.nodes[i]['importance'] for i in range(n)])
        sorted_indices = np.argsort(-importance_arr)

        print("\nskor lexrank per kalimat, urut tertinggi (tab-separated):")
        print("\t".join(["Rank", "Sentence Idx", "LexRank Score", "Sentence (truncated)"]))
        for rank, idx in enumerate(sorted_indices):
            sent_preview = sentences[idx].strip()[:70] + ("..." if len(sentences[idx].strip()) > 70 else "")
            print(f"{rank+1}\t{idx}\t{importance_arr[idx]:.6f}\t{sent_preview}")

        # simpan tsv
        tsv_path = os.path.join(REPORT_FIG_DIR, f"lexrank_scores_{article_id}.tsv")
        with open(tsv_path, "w", encoding="utf-8") as f:
            f.write("Rank\tSentence Idx\tLexRank Score\tSentence\n")
            for rank, idx in enumerate(sorted_indices):
                sent_text = sentences[idx].strip().replace("\t", " ")
                f.write(f"{rank+1}\t{idx}\t{importance_arr[idx]:.6f}\t{sent_text}\n")
        print("saved", tsv_path)

        print("\ndistribusi skor lexrank:")
        lr_stats = [
            ["Mean", f"{importance_arr.mean():.6f}"],
            ["Std Dev", f"{importance_arr.std():.6f}"],
            ["Min", f"{importance_arr.min():.6f} (S{importance_arr.argmin()})"],
            ["Max", f"{importance_arr.max():.6f} (S{importance_arr.argmax()})"],
            ["Median", f"{np.median(importance_arr):.6f}"],
            ["Sum", f"{importance_arr.sum():.6f}"],
        ]
        print(tabulate(lr_stats, headers=["Statistic", "Value"], tablefmt="grid"))

        # bar chart per kalimat
        fig, ax = plt.subplots(figsize=(max(8, n * 0.6), 5))
        colors = ['#2196F3' if importance_arr[i] >= importance_arr.mean() else '#90CAF9' for i in range(n)]
        bars = ax.bar(range(n), importance_arr, color=colors, edgecolor='black', linewidth=0.5)
        ax.axhline(importance_arr.mean(), color='red', linestyle='--', linewidth=1,
                   label=f'Mean = {importance_arr.mean():.4f}')
        ax.set_xlabel("Sentence Index")
        ax.set_ylabel("LexRank Importance Score")
        ax.set_title(f"LexRank Importance Scores - {label}")
        ax.set_xticks(range(n))
        ax.legend()
        plt.tight_layout()
        lr_bar_path = os.path.join(REPORT_FIG_DIR, f"lexrank_bar_{article_id}.png")
        plt.savefig(lr_bar_path, dpi=200)
        plt.show()
        print("saved", lr_bar_path)

        # mapping lexrank -> peluang inisialisasi (normalize(importance * bonus) * 5)
        biased_scores = importance_arr * position_bonus
        raw_probs = biased_scores / biased_scores.sum() if biased_scores.sum() > 0 else np.ones(n) / n
        init_probs = np.clip(raw_probs * 5, 0, 1)  # dipotong ke [0,1] untuk tampilan saja

        print("\nlexrank -> peluang inisialisasi, p = normalize(lexrank * bonus) * 5 (tab-separated):")
        print("\t".join(["Sentence Idx", "LexRank Score", "Position Bonus",
                         "Biased Score", "Normalized Prob", "Init Prob (×5)", "Position Role"]))
        for i in range(n):
            if i == 0:
                pos_role = "Lead sentence (2.0×)"
            elif i < 4:
                pos_role = "Early sentence (1.5×)"
            elif i < 8:
                pos_role = "Mid-early sentence (1.2×)"
            else:
                pos_role = "Later sentence (0.5×)"
            print(f"{i}\t{importance_arr[i]:.6f}\t{position_bonus[i]:.1f}\t"
                  f"{biased_scores[i]:.6f}\t{raw_probs[i]:.6f}\t{init_probs[i]:.6f}\t{pos_role}")

        # tsv
        tsv_path2 = os.path.join(REPORT_FIG_DIR, f"lexrank_init_mapping_{article_id}.tsv")
        with open(tsv_path2, "w", encoding="utf-8") as f:
            f.write("Sentence Idx\tLexRank Score\tPosition Bonus\tBiased Score\tNormalized Prob\tInit Prob (×5)\tPosition Role\n")
            for i in range(n):
                if i == 0:
                    pos_role = "Lead sentence (2.0x)"
                elif i < 4:
                    pos_role = "Early sentence (1.5x)"
                elif i < 8:
                    pos_role = "Mid-early sentence (1.2x)"
                else:
                    pos_role = "Later sentence (0.5x)"
                f.write(f"{i}\t{importance_arr[i]:.6f}\t{position_bonus[i]:.1f}\t"
                        f"{biased_scores[i]:.6f}\t{raw_probs[i]:.6f}\t{init_probs[i]:.6f}\t{pos_role}\n")
        print("saved", tsv_path2)

        # kiri: lexrank + position bonus, kanan: peluang inisialisasi
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.bar(range(n), importance_arr, color='#2196F3', edgecolor='black',
                linewidth=0.5, label='LexRank Score')
        ax1_twin = ax1.twinx()
        ax1_twin.plot(range(n), position_bonus, 'ro-', markersize=6, label='Position Bonus')
        ax1.set_xlabel("Sentence Index")
        ax1.set_ylabel("LexRank Score", color='#2196F3')
        ax1_twin.set_ylabel("Position Bonus Multiplier", color='red')
        ax1.set_title("LexRank Scores & Position Bias")
        ax1.set_xticks(range(n))
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax1_twin.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

        bar_colors = ['#4CAF50' if p > 0.5 else '#FFC107' if p > 0.2 else '#F44336' for p in init_probs]
        ax2.bar(range(n), init_probs, color=bar_colors, edgecolor='black', linewidth=0.5)
        ax2.axhline(0.5, color='gray', linestyle=':', label='50% threshold')
        ax2.set_xlabel("Sentence Index")
        ax2.set_ylabel("Initialization Probability")
        ax2.set_title("MO-WOA Population Init Probability (LexRank × 5)")
        ax2.set_xticks(range(n))
        ax2.set_ylim(0, 1.05)
        ax2.legend(fontsize=9)

        plt.tight_layout()
        combo_path = os.path.join(REPORT_FIG_DIR, f"lexrank_init_combo_{article_id}.png")
        plt.savefig(combo_path, dpi=200)
        plt.show()
        print("saved", combo_path)

    # kadang teks node rusak ("( evn )"), ganti pakai kalimat asli
    corrupted = []
    for i in range(min(5, len(graph.nodes))):
        text = graph.nodes[i].get('text', '')
        if '( evn )' in text or len(text) < 10:
            corrupted.append(i)
    if corrupted:
        for i in range(len(sentences)):
            if i in corrupted:
                graph.nodes[i]['text'] = sentences[i].strip()
                graph.nodes[i]['original_text'] = sentences[i].strip()

    # MO-WOA+
    best_solution, pareto_front, population, evaluate_obj_func, recommended_solutions, mowoa_report = mowoa_plus_algorithm_with_pareto(graph, summary_size=3, position_bonus=position_bonus, seed=0)

    selected_indices = np.where(best_solution == 1)[0]
    generated_summary_parts = []
    for s_idx in selected_indices:
        generated_summary_parts.append(graph.nodes[s_idx]['text'])
    generated_summary = ' '.join(generated_summary_parts)

    # rouge
    rouge_scores = evaluate_rouge_score(best_solution, graph, golden_summary)

    # laporan: hasil MO-WOA+
    if report_mode:
        label = f"Article {article_id}" if article_id is not None else "Article"
        n = len(sentences)
        mr = mowoa_report

        _report_header(f"MO-WOA+ - {label}")

        print("config:")
        config = [
            ["Population size", mr['pop_size']],
            ["Max iterations", mr['max_iter']],
            ["Target summary ratio", f"{mr['summary_size']*100:.0f}%"],
            ["Target sentence count", mr['target_length']],
            ["Total sentences in article", mr['num_sentences']],
            ["Redundancy ignore threshold", "0.2"],
            ["Selection method", "Knee-point over ARCHIVE (weights [32, 1])"],
            ["External archive capacity [A]", mr['archive_size']],
            ["Survival [E]", mr['survival']],
            ["Guide selection [G]", mr['guide_mode']],
            ["Transfer function [V]", mr['transfer']],
            ["LS probes per generation [LS]", mr['ls_probes']],
            ["Seed (reproducible)", mr['seed']],
            ["Function evaluations", mr['n_evaluations']],
        ]
        print(tabulate(config, headers=["Parameter", "Value"], tablefmt="grid"))

        print("\npopulasi awal (sebelum iterasi 1):")

        init_objs = mr['init_objectives']
        init_stats = [
            ["Coverage",
             f"{init_objs[:,0].mean():.4f}",
             f"{init_objs[:,0].std():.4f}",
             f"{init_objs[:,0].min():.4f}",
             f"{init_objs[:,0].max():.4f}"],
            ["Redundancy",
             f"{init_objs[:,1].mean():.4f}",
             f"{init_objs[:,1].std():.4f}",
             f"{init_objs[:,1].min():.4f}",
             f"{init_objs[:,1].max():.4f}"],
        ]
        print(tabulate(
            init_stats,
            headers=["Objective", "Mean", "Std", "Min", "Max"],
            tablefmt="grid"
        ))

        print(f"hamming diversity {mr['init_hamming_diversity']:.4f}, "
              f"korelasi biased_imp vs frekuensi seleksi {mr['init_freq_importance_corr']:+.4f}")

        # kalimat yang paling sering muncul di populasi awal
        K = min(5, mr['num_sentences'])
        print(f"\ntop-{K} kalimat paling sering dipilih di populasi awal (tab-separated):")
        header = ["Rank", "Sent Idx", "Selection Freq",
                  "LexRank", "Position Bonus", "Biased Imp"]
        print("\t".join(header))
        freq = mr['init_selection_frequency']
        topk_idx = np.argsort(-freq)[:K]
        for rank, si in enumerate(topk_idx, 1):
            print(f"{rank}\t{si}\t{freq[si]:.3f}\t"
                  f"{mr['node_importance'][si]:.4f}\t"
                  f"{mr['position_bonus'][si]:.2f}\t"
                  f"{mr['biased_importance'][si]:.4f}")

        # tsv
        init_tsv_path = os.path.join(REPORT_FIG_DIR, f"init_pop_{article_id}.tsv")
        with open(init_tsv_path, "w", encoding="utf-8") as f:
            f.write("Objective\tMean\tStd\tMin\tMax\n")
            for row in init_stats:
                f.write("\t".join(row) + "\n")
            f.write(f"\nNormalized Hamming Diversity\t{mr['init_hamming_diversity']:.4f}\n")
            f.write(f"Biased Importance Correlation\t{mr['init_freq_importance_corr']:.4f}\n")
            f.write("\nRank\tSent Idx\tSelection Freq\tLexRank\tPosition Bonus\tBiased Imp\n")
            for rank, si in enumerate(topk_idx, 1):
                f.write(f"{rank}\t{si}\t{freq[si]:.3f}\t"
                        f"{mr['node_importance'][si]:.4f}\t"
                        f"{mr['position_bonus'][si]:.2f}\t"
                        f"{mr['biased_importance'][si]:.4f}\n")
        print("saved", init_tsv_path)

        # frekuensi seleksi vs biased importance
        fig, ax = plt.subplots(figsize=(max(8, mr['num_sentences'] * 0.6), 4.5))
        x_axis = np.arange(mr['num_sentences'])
        ax.bar(x_axis - 0.2, mr['init_selection_frequency'],
               width=0.4, color='#2196F3', edgecolor='black', linewidth=0.5,
               label='Init selection frequency')
        bi = mr['biased_importance']
        bi_norm = bi / bi.max() if bi.max() > 0 else bi
        ax.bar(x_axis + 0.2, bi_norm,
               width=0.4, color='#FF9800', edgecolor='black', linewidth=0.5,
               label='Biased importance (normalized)')
        ax.set_xlabel("Sentence Index")
        ax.set_ylabel("Value")
        ax.set_title(f"Initial Population: Selection Frequency vs Biased Importance - {label}")
        ax.set_xticks(x_axis)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        init_fig_path = os.path.join(REPORT_FIG_DIR, f"init_pop_freq_{article_id}.png")
        plt.savefig(init_fig_path, dpi=200)
        plt.show()
        print("saved", init_fig_path)

        conv = mr['convergence_history']
        print(f"\nfitness {conv[0]:.6f} -> {conv[-1]:.6f} ({conv[-1] - conv[0]:+.6f}), "
              f"konvergen di iterasi {np.argmax(conv == conv[-1]) + 1}/{mr['max_iter']}")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(range(1, len(conv)+1), conv, 'b-', linewidth=1.5)
        ax1.set_xlabel("Iteration")
        ax1.set_ylabel("Best Fitness (scalar)")
        ax1.set_title(f"Fitness Convergence - {label}")
        ax1.grid(True, alpha=0.3)

        psz = mr['pareto_size_history']
        ax2.plot(range(1, len(psz)+1), psz, 'g-o', markersize=3, linewidth=1)
        ax2.set_xlabel("Iteration")
        ax2.set_ylabel("Pareto Front Size")
        ax2.set_title(f"Pareto Front Size Over Iterations - {label}")
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        conv_path = os.path.join(REPORT_FIG_DIR, f"convergence_{article_id}.png")
        plt.savefig(conv_path, dpi=200)
        plt.show()
        print("saved", conv_path)

        print("\ndinamika archive per iterasi:")
        lh = mr['leader_history']
        n_iters = len(lh)
        unique_leaders = len(set(r['leader_idx'] for r in lh))
        boundary_count = sum(1 for r in lh if r['is_boundary'])
        boundary_pct = 100.0 * boundary_count / n_iters if n_iters > 0 else 0.0
        churn_pct = 100.0 * unique_leaders / n_iters if n_iters > 0 else 0.0

        leader_stats = [
            ["Total iterations",                   n_iters],
            ["Distinct archive representatives",   unique_leaders],
            ["Representative churn rate",          f"{churn_pct:.1f}%"],
            ["Iterations w/ boundary repr.",       f"{boundary_count} ({boundary_pct:.1f}%)"],
            ["Avg archive size over run",
             f"{np.mean([r['pareto_size'] for r in lh]):.1f}"],
        ]
        print(tabulate(leader_stats,
                       headers=["Statistic", "Value"],
                       tablefmt="grid"))

        fig, (axL1, axL2) = plt.subplots(1, 2, figsize=(11, 4))
        iters_ax = [r['iter'] + 1 for r in lh]

        axL1.fill_between(iters_ax,
                          [r['cov_min'] for r in lh],
                          [r['cov_max'] for r in lh],
                          color='tab:blue', alpha=0.18,
                          label='Archive extent (min–max)')
        axL1.plot(iters_ax, [r['cov_max'] for r in lh], 'b-', linewidth=1)
        axL1.plot(iters_ax, [r['cov_min'] for r in lh], 'b-', linewidth=1)
        axL1.plot(iters_ax, [r['knee_cov'] for r in lh], 'k--o',
                  markersize=2.5, linewidth=1, label='Knee point')
        axL1.set_xlabel("Iteration"); axL1.set_ylabel("Coverage")
        axL1.set_title("Archive Coverage Extent & Knee")
        axL1.grid(True, alpha=0.3); axL1.legend(fontsize=8)

        axL2.fill_between(iters_ax,
                          [r['red_min'] for r in lh],
                          [r['red_max'] for r in lh],
                          color='tab:red', alpha=0.18,
                          label='Archive extent (min–max)')
        axL2.plot(iters_ax, [r['red_max'] for r in lh], 'r-', linewidth=1)
        axL2.plot(iters_ax, [r['red_min'] for r in lh], 'r-', linewidth=1,
                  label='Lowest redundancy reached')
        axL2.plot(iters_ax, [r['knee_red'] for r in lh], 'k--o',
                  markersize=2.5, linewidth=1, label='Knee point')
        axL2.set_xlabel("Iteration"); axL2.set_ylabel("Redundancy")
        axL2.set_title("Archive Redundancy Extent & Knee")
        axL2.grid(True, alpha=0.3); axL2.legend(fontsize=8)

        plt.suptitle(f"Archive Front Dynamics - {label}",
                     fontsize=12, y=1.02)
        plt.tight_layout()
        leader_fig_path = os.path.join(REPORT_FIG_DIR,
                                       f"leader_dynamics_{article_id}.png")
        plt.savefig(leader_fig_path, dpi=200, bbox_inches='tight')
        plt.show()
        print("saved", leader_fig_path)

        leader_tsv = os.path.join(REPORT_FIG_DIR,
                                  f"leader_history_{article_id}.tsv")
        with open(leader_tsv, "w", encoding="utf-8") as f:
            f.write("Iter\tRepr Idx\tCoverage\tRedundancy\t"
                    "Crowding Dist\tBoundary?\tArchive Size\t"
                    "CovMin\tCovMax\tRedMin\tRedMax\t"
                    "KneeCov\tKneeRed\n")
            for r in lh:
                cd_str = "inf" if np.isinf(r['crowding_distance']) \
                         else f"{r['crowding_distance']:.4f}"
                bnd = "Yes" if r['is_boundary'] else "No"
                f.write(f"{r['iter']+1}\t{r['leader_idx']}\t"
                        f"{r['coverage']:.4f}\t{r['redundancy']:.4f}\t"
                        f"{cd_str}\t{bnd}\t{r['pareto_size']}\t"
                        f"{r['cov_min']:.4f}\t{r['cov_max']:.4f}\t"
                        f"{r['red_min']:.4f}\t{r['red_max']:.4f}\t"
                        f"{r['knee_cov']:.4f}\t{r['knee_red']:.4f}\n")
        print("saved", leader_tsv)

        print(f"\nfront pareto: {len(mr['final_front'])} solusi")

        if mr['pareto_objs'] is not None and len(mr['pareto_objs']) > 0:
            po = mr['pareto_objs']
            print(f"coverage {po[:,0].min():.4f}-{po[:,0].max():.4f}, "
                  f"redundancy {po[:,1].min():.4f}-{po[:,1].max():.4f}")

            print("\nsolusi front, urut coverage turun (tab-separated):")
            header = ["#", "Coverage", "Redundancy",
                      "Norm Distance", "Selected?"]
            print("\t".join(header))

            sorted_idx = np.argsort(-po[:, 0])
            for rank, pi in enumerate(sorted_idx):
                is_selected = "★ KNEE-POINT" if pi == mr['knee_idx_in_pareto'] else ""
                dist_val = f"{mr['pareto_distances'][pi]:.4f}" if mr['pareto_distances'] is not None else "—"
                print(f"{rank+1}\t{po[pi,0]:.6f}\t{po[pi,1]:.6f}\t{dist_val}\t{is_selected}")

            tsv_path = os.path.join(REPORT_FIG_DIR, f"pareto_front_{article_id}.tsv")
            with open(tsv_path, "w", encoding="utf-8") as f:
                f.write("\t".join(header) + "\n")
                for rank, pi in enumerate(sorted_idx):
                    is_selected = "KNEE-POINT" if pi == mr['knee_idx_in_pareto'] else ""
                    dist_val = f"{mr['pareto_distances'][pi]:.4f}" if mr['pareto_distances'] is not None else ""
                    f.write(f"{rank+1}\t{po[pi,0]:.6f}\t{po[pi,1]:.6f}\t{dist_val}\t{is_selected}\n")
            print("saved", tsv_path)

            # plot coverage vs redundancy
            fig, ax = plt.subplots(figsize=(7, 5))

            ax.scatter(po[:, 0], po[:, 1], c='steelblue', s=60, edgecolors='black',
                       linewidths=0.5, zorder=2, label='Pareto solutions')
            if mr['knee_idx_in_pareto'] is not None:
                kp = mr['knee_idx_in_pareto']
                ax.scatter(po[kp, 0], po[kp, 1], c='gold', s=200, marker='*',
                           edgecolors='black', linewidths=1, zorder=3, label='Knee-point')
            ax.set_xlabel("Coverage ↑")
            ax.set_ylabel("Redundancy ↓")
            ax.set_title("Pareto Front: Coverage vs Redundancy")
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)

            plt.suptitle(f"Pareto Front 2D Projections - {label}", fontsize=13, y=1.02)
            plt.tight_layout()
            pareto_path = os.path.join(REPORT_FIG_DIR, f"pareto_2d_{article_id}.png")
            plt.savefig(pareto_path, dpi=200, bbox_inches='tight')
            plt.show()
            print("saved", pareto_path)

            print("\nknee point vs solusi ekstrem (tab-separated):")
            kp = mr['knee_idx_in_pareto']
            distances = mr['pareto_distances']
            knee_dist = distances[kp]
            knee_obj = po[kp]

            best_cov_idx = int(np.argmax(po[:, 0]))
            min_red_idx  = int(np.argmin(po[:, 1]))

            extremes = [
                ("Knee-point (chosen)",        kp),
                ("Best coverage extreme",      best_cov_idx),
                ("Lowest redundancy extreme",  min_red_idx),
            ]

            print("Selection Strategy\tCoverage\tRedundancy\tDist. to Ideal")
            for name, idx in extremes:
                marker = " ★" if idx == kp else ""
                print(f"{name}{marker}\t{po[idx,0]:.4f}\t{po[idx,1]:.4f}\t"
                      f"{distances[idx]:.4f}")

            # posisi jarak knee point dibanding solusi lain di front
            sorted_dist = np.sort(distances)
            knee_rank = int(np.where(sorted_dist == knee_dist)[0][0]) + 1
            front_size = len(distances)
            dist_median = float(np.median(distances))
            dist_mean   = float(np.mean(distances))
            dist_max    = float(np.max(distances))

            print(f"\njarak knee ke ideal {knee_dist:.4f} (median {dist_median:.4f}, "
                  f"mean {dist_mean:.4f}, max {dist_max:.4f}), rank {knee_rank}/{front_size}, "
                  f"{(1.0 - knee_dist/dist_max)*100:.1f}% lebih dekat dari solusi terjauh")

        print("\nsolusi terpilih (knee point):")
        best_cov, best_red = mr['best_objectives']
        sel_table = [
            ["Selected solution index (in population)", mr['best_idx']],
            ["Coverage (biased LexRank sum)", f"{best_cov:.6f}"],
            ["Redundancy", f"{best_red:.6f}"],
            ["Number of sentences selected", int(mr['best_solution'].sum())],
            ["Target sentence count (hard constraint)", mr['target_length']],
        ]
        if mr['pareto_distances'] is not None and mr['knee_idx_in_pareto'] is not None:
            sel_table.append(["Euclidean distance to ideal", f"{mr['pareto_distances'][mr['knee_idx_in_pareto']]:.6f}"])

        total_biased = mr['biased_importance'].sum()
        sel_table.append(["Coverage ratio (selected / total)", f"{best_cov / total_biased * 100:.1f}%"])
        print(tabulate(sel_table, headers=["Property", "Value"], tablefmt="grid"))

        print("\nkalimat terpilih (tab-separated):")
        print("\t".join(["Sentence Idx", "LexRank Score", "Position Bonus",
                         "Biased Score", "Sentence Text"]))
        for s_idx in selected_indices:
            lr = graph.nodes[s_idx]['importance']
            pb = position_bonus[s_idx]
            bs = mr['biased_importance'][s_idx]
            txt = graph.nodes[s_idx]['text'].replace("\t", " ")
            print(f"{s_idx}\t{lr:.6f}\t{pb:.1f}\t{bs:.6f}\t{txt}")

        tsv_path = os.path.join(REPORT_FIG_DIR, f"selected_sentences_{article_id}.tsv")
        with open(tsv_path, "w", encoding="utf-8") as f:
            f.write("Sentence Idx\tLexRank Score\tPosition Bonus\tBiased Score\tSentence Text\n")
            for s_idx in selected_indices:
                lr = graph.nodes[s_idx]['importance']
                pb = position_bonus[s_idx]
                bs = mr['biased_importance'][s_idx]
                txt = graph.nodes[s_idx]['text'].replace("\t", " ")
                f.write(f"{s_idx}\t{lr:.6f}\t{pb:.1f}\t{bs:.6f}\t{txt}\n")
        print("saved", tsv_path)

        _report_header(f"Hasil ringkasan - {label}")
        print(f"generated ({len(selected_indices)} kalimat):\n{generated_summary}")
        print(f"\nreference:\n{golden_summary}")
        cr = len(generated_summary) / max(1, sum(len(s) for s in sentences))
        print(f"\npanjang artikel {sum(len(s) for s in sentences)}, generated {len(generated_summary)}, "
              f"reference {len(golden_summary)} karakter, CR {cr:.1%}")

        print("\nROUGE:")
        rouge_table = []
        for metric_name in ['rouge1', 'rouge2', 'rougeL']:
            s = rouge_scores[metric_name]
            rouge_table.append([metric_name.upper(), f"{s.precision:.4f}",
                                f"{s.recall:.4f}", f"{s.fmeasure:.4f}"])
        print(tabulate(rouge_table, headers=["Metric", "Precision", "Recall", "F1-Score"],
                       tablefmt="grid"))

    return rouge_scores, generated_summary, embedding_report, mowoa_report

In [ ]:
# paragraphs = train_df.paragraphs[1]
# proprocessed_paragraphs = preprocess_paragraph(paragraphs)
# golden_summary = tokens_to_narrative(train_df.summary[1])

# print(paragraphs)
# print(proprocessed_paragraphs)
# print(golden_summary)

Single article run (comment if not needed):

In [ ]:
# current_golden_summary = tokens_to_narrative(train_df.summary[1])

# current_article_rouge_score, generated_summary, embedding_report, mowoa_report = summarize(train_df.paragraphs[1], current_golden_summary, article_id=2)

In [ ]:
# print(generated_summaries[0])
# print(golden_summaries[0])

In [ ]:
!pip install bert-score -q

import torch
import bert_score
import numpy as np
from typing import List, Tuple, Union
import pandas as pd

def compute_bertscore_metrics(
    candidates: List[str],
    references: List[str],
    model_type: str = 'indobenchmark/indobert-large-p2',
    batch_size: int = 8,
    device: str = None,
    verbose: bool = True
) -> dict:
    """BERTScore rata-rata + per ringkasan. model default indobert-large-p2"""
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    print(f"bertscore {model_type} ({device}), {len(candidates)} cand / {len(references)} ref")

    # indobert-large-p2 punya 24 layer, harus diset manual
    num_layers = 24

    P, R, F1 = bert_score.score(
        candidates, references,
        model_type=model_type,
        num_layers=num_layers,
        batch_size=batch_size,
        device=device,
        return_hash=False,
        lang='id',
        verbose=verbose,
    )

    P_np, R_np, F1_np = P.detach().cpu().numpy(), R.detach().cpu().numpy(), F1.detach().cpu().numpy()

    results = {
        'avg_precision': P_np.mean(),
        'avg_recall': R_np.mean(),
        'avg_f1': F1_np.mean(),
        'precision_scores': P_np,
        'recall_scores': R_np,
        'f1_scores': F1_np,
        'model_used': model_type,
        'device_used': device
    }

    if verbose:
        print(f"BERTScore P={results['avg_precision']:.4f} R={results['avg_recall']:.4f} "
              f"F1={results['avg_f1']:.4f}")

    return results


In [ ]:
# bert_results = compute_bertscore_metrics(generated_summaries, golden_summaries)

# n-gram redundancy score evaluation

In [ ]:
import re
from collections import Counter
from typing import List
import pandas as pd
import numpy as np

def compute_ngram_redundancy(summary: str, n: int = 3, threshold: float = 0.3) -> float:
  """skor redundansi n-gram (0 = tidak redundan). threshold dikurangkan dari hasil"""
  words = re.findall(r'\b\w+\b', summary.lower())
  if len(words) < n:
    return 0.0

  ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]

  ngram_counts = Counter(ngrams)
  total_ngrams = len(ngrams)

  if total_ngrams == 0:
    return 0.0

  max_freq = max(ngram_counts.values()) if ngram_counts else 0

  # proporsi n-gram yang paling sering muncul
  redundancy = max_freq / total_ngrams

  return max(0.0, redundancy - threshold)

def compute_ngram_redundancy_batch(summaries: List[str], n: int = 3, threshold: float = 0.3) -> dict:
  """versi batch dari compute_ngram_redundancy"""
  redundancy_scores = []

  print(f"Computing {n}-gram redundancy for {len(summaries)} summaries...")

  for i, summary in enumerate(summaries):
      score = compute_ngram_redundancy(summary, n=n, threshold=threshold)
      redundancy_scores.append(score)

      if (i+1) % 10 == 0:
          print(f"   Processed {i+1}/{len(summaries)} summaries")

  avg_redundancy = np.mean(redundancy_scores)

  results = {
      'avg_redundancy': avg_redundancy,
      'redundancy_scores': np.array(redundancy_scores),
      'n_gram': n,
      'threshold': threshold
  }

  print(f"{n}-gram redundancy (threshold {threshold}): mean {avg_redundancy:.4f}, "
        f"range {np.min(redundancy_scores):.4f}-{np.max(redundancy_scores):.4f}")

  return results

In [ ]:
# compute_ngram_redundancy_batch(generated_summaries)

# Calculate compression ratio

In [ ]:
def compute_compression_ratio(
    source_texts: List[str],
    summary_texts: List[str]
) -> dict:
    """CR = panjang ringkasan / panjang sumber (karakter)"""
    source_lengths = [len(text) for text in source_texts]
    summary_lengths = [len(text) for text in summary_texts]

    compression_ratios = np.array(summary_lengths) / np.array(source_lengths)

    results = {
        'avg_compression_ratio': np.mean(compression_ratios),
        'compression_ratios': compression_ratios,
        'avg_source_length': np.mean(source_lengths),
        'avg_summary_length': np.mean(summary_lengths),
        'source_lengths': source_lengths,
        'summary_lengths': summary_lengths
    }

    print(f"avg source {results['avg_source_length']:,.0f} chars, "
          f"avg summary {results['avg_summary_length']:,.0f} chars")
    print(f"CR mean {results['avg_compression_ratio']:.1%}, "
          f"range {np.min(compression_ratios):.1%}-{np.max(compression_ratios):.1%}")

    return results

In [ ]:
# source_texts = []
# for paragraphs in train_df.paragraphs[:len(generated_summaries)]:
#     # Use the same sentence reconstruction as split_text_into_sentences
#     sentences = []
#     for paragraph in paragraphs:
#         for sentence_tokens in paragraph:
#             sentence = ' '.join(sentence_tokens)
#             sentence = sentence.replace(' ,', ',').replace(' .', '.')
#             sentence = sentence.replace(' :', ':').replace(' ;', ';')
#             sentence = sentence.replace(' !', '!').replace(' ?', '?')
#             if sentence.strip():
#                 sentences.append(sentence.strip())
#     source_texts.append(' '.join(sentences))

# compute_compression_ratio(source_texts, generated_summaries)

# Implementation Loop

In [ ]:
# loop eksperimen 150 artikel
# pass 1: jalankan semua tanpa gambar, simpan diagnostik per artikel
# pass 2: pilih satu artikel untuk gambar laporan (bab IV 4.4.2-4.4.3), skor berbobot:
#   red_descent  redundansi terendah archive turun (efek LS probe)
#   conv_gain    fitness naik, biar grafik konvergensi tidak datar
#   rouge_repr   rouge-1 dekat rata-rata korpus
#   front_size   archive akhir besar
#   growth       archive baru penuh di akhir
#   red_late     redundansi masih turun di iterasi akhir
#   cr_band      compression ratio dekat 15-25%
#   cov_gain     coverage maksimum naik selama pencarian
#   lexrank_cv   lexrank bervariasi (tidak cuma ikut posisi)
# hanya artikel "sulit" (C(N,3) > 1480), artikel kecil semua kombinasi pasti ketemu
# REPORT_ARTICLE_IDX: 'auto' / index 0-based / None (tanpa gambar)
from math import comb

REPORT_ARTICLE_IDX = 'auto'
REPORT_WEIGHTS = dict(red_descent=0.18, conv_gain=0.16, rouge_repr=0.14,
                      front_size=0.14, growth=0.12, red_late=0.10,
                      cr_band=0.07, cov_gain=0.06, lexrank_cv=0.03)
FE_BUDGET = 1480

report_mode = False          # global, dibaca summarize()

generated_summaries = []
golden_summaries = []
source_texts = []

rouge1_scores = {'precision': [], 'recall': [], 'f1': []}
rouge2_scores = {'precision': [], 'recall': [], 'f1': []}
rougel_scores = {'precision': [], 'recall': [], 'f1': []}

diagnostics = []             # satu dict per artikel

# NUM_ARTICLES = len(train_df.paragraphs)
NUM_ARTICLES = 150
# NUM_ARTICLES = 1

# pass 1
_warned_keys = False

for idx in range(NUM_ARTICLES):
    current_paragraphs = test_df.paragraphs[idx]
    current_golden_summary = tokens_to_narrative(test_df.summary[idx])

    report_mode = False
    current_rouge, gen_summary, _emb_rep, mowoa_report = \
        summarize(current_paragraphs, current_golden_summary, article_id=idx+1)

    for metric_key, score_dict in [('rouge1', rouge1_scores),
                                    ('rouge2', rouge2_scores),
                                    ('rougeL', rougel_scores)]:
        score_dict['precision'].append(current_rouge[metric_key].precision)
        score_dict['recall'].append(current_rouge[metric_key].recall)
        score_dict['f1'].append(current_rouge[metric_key].fmeasure)

    generated_summaries.append(gen_summary)
    golden_summaries.append(current_golden_summary)

    sentences = split_text_into_sentences(current_paragraphs)
    source_text = ' '.join(sentences)
    source_texts.append(source_text)

    # ambil angka2 untuk pemilihan artikel, report-nya dibuang
    lh    = mowoa_report.get('leader_history', [])
    front = mowoa_report.get('final_front', [])
    conv  = np.asarray(mowoa_report.get('convergence_history', []), dtype=float)
    ni    = np.asarray(mowoa_report.get('node_importance', [0.0]), dtype=float)
    n_sen = int(mowoa_report.get('num_sentences', len(sentences)))

    sizes    = [r.get('pareto_size', np.nan) for r in lh]
    red_mins = [r.get('red_min',     np.nan) for r in lh]
    cov_maxs = [r.get('cov_max',     np.nan) for r in lh]

    if lh and not np.isfinite(red_mins[0]) and not _warned_keys:
        print("warning: leader_history tidak punya red_min/cov_max (cell algoritma versi lama?)\n")
        _warned_keys = True

    n_it = max(1, len(sizes) - 1)
    if len(sizes) >= 2 and np.isfinite(sizes[-1]):
        first_at    = next((i for i, s in enumerate(sizes) if s >= sizes[-1]), n_it)
        growth_frac = first_at / n_it
        had_dip     = any(sizes[i+1] < sizes[i] for i in range(len(sizes)-1))
    else:
        growth_frac, had_dip = np.nan, False

    # perubahan relatif
    def _rel_drop(seq):
        if len(seq) < 2 or not (np.isfinite(seq[0]) and np.isfinite(seq[-1])):
            return np.nan
        return (seq[0] - seq[-1]) / (abs(seq[0]) + 1e-12)

    def _rel_rise(seq):
        if len(seq) < 2 or not (np.isfinite(seq[0]) and np.isfinite(seq[-1])):
            return np.nan
        return (seq[-1] - seq[0]) / (abs(seq[-1]) + 1e-12)

    red_descent = _rel_drop(red_mins)
    cov_gain    = _rel_rise(cov_maxs)
    conv_gain   = _rel_rise(list(conv)) if conv.size >= 2 else np.nan

    last_imp = 0
    for i in range(1, len(red_mins)):
        if (np.isfinite(red_mins[i]) and np.isfinite(red_mins[i-1])
                and red_mins[i] < red_mins[i-1] - 1e-9):
            last_imp = i
    red_late = last_imp / n_it if len(red_mins) >= 2 else np.nan

    diagnostics.append(dict(
        idx=idx, N=n_sen, n_combos=comb(n_sen, 3) if n_sen >= 3 else 0,
        front_size=len(front), red_descent=red_descent, red_late=red_late,
        conv_gain=conv_gain, cov_gain=cov_gain, growth_frac=growth_frac,
        dip=1.0 if had_dip else 0.0,
        lexrank_cv=float(ni.std() / ni.mean()) if ni.mean() > 0 else np.nan,
        rouge1=current_rouge['rouge1'].fmeasure,
        cr=len(gen_summary) / max(1, len(source_text)),
    ))
    del mowoa_report, _emb_rep          # hemat memori colab

    if (idx + 1) % 50 == 0:
        print(f"Processed {idx+1}/{NUM_ARTICLES} articles...")

print(f"selesai, {NUM_ARTICLES} artikel")

# pass 2: skor artikel, buat ulang gambar untuk artikel terpilih
def _norm(vals, invert=False):
    v = np.asarray(vals, dtype=float)
    ok = np.isfinite(v)
    out = np.zeros(len(v))
    if not ok.any():
        return out
    lo, hi = v[ok].min(), v[ok].max()
    out[ok] = 0.5 if (hi - lo) < 1e-12 else (v[ok] - lo) / (hi - lo)
    return np.where(ok, 1.0 - out, 0.0) if invert else out

if REPORT_ARTICLE_IDX is None:
    report_idx = -1
elif REPORT_ARTICLE_IDX != 'auto':
    report_idx = int(REPORT_ARTICLE_IDX)
else:
    hard = [d for d in diagnostics if d['n_combos'] > FE_BUDGET]
    pool = hard if len(hard) >= 3 else diagnostics
    print(f"\nSelecting the report article from {len(pool)} "
          f"{'HARD' if pool is hard else 'ALL'} articles "
          f"(hard = C(N,3) > {FE_BUDGET}).")

    r1_mean = float(np.mean([d['rouge1'] for d in diagnostics]))
    cr_pen = [0.0 if 0.15 <= d['cr'] <= 0.25
              else min(abs(d['cr'] - 0.15), abs(d['cr'] - 0.25)) for d in pool]
    comp = {
        'red_descent': _norm([d['red_descent'] for d in pool]),
        'conv_gain':   _norm([d['conv_gain']   for d in pool]),
        'front_size':  _norm([d['front_size']  for d in pool]),
        'growth':      _norm([d['growth_frac'] for d in pool]),
        'red_late':    _norm([d['red_late']    for d in pool]),
        'cov_gain':    _norm([d['cov_gain']    for d in pool]),
        'lexrank_cv':  _norm([d['lexrank_cv']  for d in pool]),
        'rouge_repr':  _norm([abs(d['rouge1'] - r1_mean) for d in pool], invert=True),
        'cr_band':     _norm(cr_pen, invert=True),
    }
    total = np.zeros(len(pool))
    for k, w in REPORT_WEIGHTS.items():
        total += w * comp[k]
    for i, d in enumerate(pool):
        d['score'] = float(total[i])
        flags = []
        if not (d['conv_gain'] > 0.02):     flags.append('flatConv')
        if not (d['red_late'] > 0.30):      flags.append('earlyRed')
        if d['cr'] < 0.10:                  flags.append('lowCR')
        if abs(d['rouge1'] - r1_mean) > 0.25: flags.append('outlierR1')
        d['flags'] = ','.join(flags) if flags else '-'

    ranked = sorted(pool, key=lambda d: -d['score'])
    print(f"corpus mean ROUGE-1 F1 = {r1_mean:.4f}\n")
    print(f"{'#':>3}{'idx':>5}{'N':>4}{'|arch|':>7}{'redDrop':>8}{'redLate':>8}"
          f"{'convRise':>9}{'covRise':>8}{'growth':>7}{'dip':>4}{'R-1':>7}"
          f"{'CR':>7}{'score':>7}  flags")
    for r, d in enumerate(ranked[:12], 1):
        print(f"{r:>3}{d['idx']:>5}{d['N']:>4}{d['front_size']:>7}"
              f"{d['red_descent']:>8.3f}{d['red_late']:>8.2f}{d['conv_gain']:>9.3f}"
              f"{d['cov_gain']:>8.3f}{d['growth_frac']:>7.2f}"
              f"{'y' if d['dip'] else '-':>4}{d['rouge1']:>7.3f}"
              f"{100*d['cr']:>6.1f}%{d['score']:>7.3f}  {d['flags']}")

    report_idx = ranked[0]['idx']
    w = ranked[0]
    print(f"\n-> selected idx {report_idx} (article_id {report_idx+1}): "
          f"N = {w['N']}, C(N,3) = {w['n_combos']} "
          f"({w['n_combos']/FE_BUDGET:.1f}x the evaluation budget), "
          f"archive = {w['front_size']} solutions, redundancy drop "
          f"{100*w['red_descent']:.1f}% (last improvement at iter "
          f"{w['red_late']*50:.0f}), scalar rise {100*w['conv_gain']:.1f}%, "
          f"ROUGE-1 = {w['rouge1']:.4f}, CR = {100*w['cr']:.1f}%.")
    if w['flags'] != '-':
        print(f"   flags: {w['flags']} -> cek tabel di atas, set REPORT_ARTICLE_IDX kalau mau ganti")

if report_idx >= 0:
    print(f"\nbuat ulang gambar untuk idx {report_idx} (seeded, hasil sama)\n")
    report_mode = True
    _r, _g, _e, REPORT_MR = summarize(test_df.paragraphs[report_idx],
                                      tokens_to_narrative(test_df.summary[report_idx]),
                                      article_id=report_idx + 1)
    REPORT_IDX = report_idx
    report_mode = False

### Output Bab IV figures set

In [ ]:
# gambar & tabel bab IV 4.4.2-4.4.3 untuk artikel terpilih
# semua C(N,3) kombinasi dienumerasi -> front Pareto P* yang sebenarnya diketahui,
# jadi konvergensi bisa dihitung sebagai % dari optimum dan recovery dicek per titik
# output: gambar A-E, tabel 1-3
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

def plot_pareto_curve(front_xy, color='k', label=None, lw=1.6,
                      style='-', marker_size=6, dashed_only=False):
    """kurva PCHIP (monoton) lewat titik front + marker"""
    P = np.atleast_2d(np.asarray(front_xy, float))
    P = P[np.argsort(P[:, 0])]
    _, u = np.unique(np.round(P[:, 0], 12), return_index=True)
    P = P[np.sort(u)]
    if len(P) >= 3:
        f  = PchipInterpolator(P[:, 0], P[:, 1])
        xs = np.linspace(P[0, 0], P[-1, 0], 400)
        plt.plot(xs, f(xs), style, color=color, lw=lw, zorder=2,
                 alpha=(0.5 if dashed_only else 1.0), label=label)
    else:
        plt.plot(P[:, 0], P[:, 1], style, color=color, lw=lw, zorder=2,
                 alpha=(0.5 if dashed_only else 1.0), label=label)
    if not dashed_only:
        plt.scatter(P[:, 0], P[:, 1], s=marker_size**2, c=color,
                    edgecolors='white', linewidths=0.9, zorder=3)
    return P

mr   = REPORT_MR
bi   = np.asarray(mr['biased_importance'], float)
adj  = np.asarray(mr['adj_matrix'], float)
lh   = mr['leader_history']
N    = int(mr['num_sentences'])
K    = int(mr['target_length'])
AID  = REPORT_IDX + 1

# enumerasi semua solusi feasible
combos  = np.array(list(combinations(range(N), K)), dtype=int)
cov_all = bi[combos].sum(axis=1)
iu, ju  = np.triu_indices(K, k=1)
npairs  = K * (K - 1) // 2
red_all = np.maximum(0.0, adj[combos[:, iu], combos[:, ju]].sum(axis=1)
                          / npairs - 0.2)
F_min_all = np.column_stack([-cov_all, red_all])

def nd_mask(F):
    keep = np.ones(len(F), bool)
    for i in range(len(F)):
        if not keep[i]:
            continue
        if np.any(np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)):
            keep[i] = False
    return keep

P_star = np.unique(np.round(F_min_all[nd_mask(F_min_all)], 12), axis=0)
Ps     = P_star[np.argsort(P_star[:, 0])]

from rouge_score import rouge_scorer
sents_rep = split_text_into_sentences(test_df.paragraphs[REPORT_IDX])
gold_rep  = tokens_to_narrative(test_df.summary[REPORT_IDX])
assert len(sents_rep) == N, f"sentence list {len(sents_rep)} != N {N}"

_sc = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)
r1 = np.array([_sc.score(gold_rep, ' '.join(sents_rep[i] for i in sorted(c)))
               ['rouge1'].fmeasure for c in combos])

_on = np.array([np.any(np.all(np.isclose(P_star, F_min_all[j]), axis=1))
                for j in range(len(combos))])
i_oracle = int(np.argmax(r1))
i_front  = int(np.argmax(np.where(_on, r1, -1)))
i_knee   = int(np.argmin(np.hypot(cov_all - lh[-1]['knee_cov'],
                                  red_all - lh[-1]['knee_red'])))
print(f"oracle {r1[i_oracle]:.4f} (on front: {_on[i_oracle]}) | "
      f"best-on-front {r1[i_front]:.4f} | delivered {r1[i_knee]:.4f}")

print(f"Enumerasi C({N},{K}) = {len(combos):,} kandidat feasible "
      f"({len(combos)/1480:.1f}x budget 1480 evaluasi)")
print(f"Front Pareto sebenarnya |P*| = {len(P_star)} solusi")

# normalisasi pakai seluruh ruang feasible
c_lo, c_hi = cov_all.min(), cov_all.max()
r_lo, r_hi = red_all.min(), red_all.max()
c_rng = max(c_hi - c_lo, 1e-12)
r_rng = max(r_hi - r_lo, 1e-12)
REF   = np.array([1.1, 1.1])

def to_norm(F_min):
    F_min = np.atleast_2d(np.asarray(F_min, float))
    return np.column_stack([(c_hi - (-F_min[:, 0])) / c_rng,
                            (F_min[:, 1] - r_lo) / r_rng])

def hv2d(S, ref=REF):
    S = np.atleast_2d(S)
    S = S[np.all(S <= ref, axis=1)]
    if len(S) == 0:
        return 0.0
    S = np.unique(np.round(S[nd_mask(S)], 12), axis=0)
    S = S[np.argsort(S[:, 0])]
    hv, prev = 0.0, ref[1]
    for f1, f2 in S:
        if f2 < prev:
            hv += (ref[0] - f1) * (prev - f2)
            prev = f2
    return hv

def as_min(arch):                      # archive disimpan (coverage, redundancy)
    arch = np.atleast_2d(np.asarray(arch, float))
    return np.column_stack([-arch[:, 0], arch[:, 1]])

hv_star = hv2d(to_norm(P_star))
iters   = [r['iter'] + 1 for r in lh]
sizes   = [r['pareto_size'] for r in lh]
hv_pct  = [100.0 * hv2d(to_norm(as_min(r['arch_front']))) / hv_star for r in lh]

rec_pct = []
for r in lh:
    A   = np.unique(np.round(as_min(r['arch_front']), 12), axis=0)
    hit = sum(1 for f in A if np.any(np.all(np.isclose(P_star, f), axis=1)))
    rec_pct.append(100.0 * hit / len(P_star))

fin      = np.asarray(lh[-1]['arch_front'], float)
fin_uni  = np.unique(np.round(as_min(fin), 12), axis=0)
rec_fin  = sum(1 for f in fin_uni
               if np.any(np.all(np.isclose(P_star, f), axis=1)))

# iterasi pertama kali tiap titik archive akhir masuk ke archive
disc = []
for pt in fin:
    first = len(lh)
    for t, r in enumerate(lh):
        if np.any(np.all(np.isclose(np.asarray(r['arch_front'], float), pt,
                                    atol=1e-12), axis=1)):
            first = t
            break
    disc.append(first + 1)
disc = np.array(disc)

# gambar A: konvergensi hypervolume + ukuran archive
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
a1.plot(iters, hv_pct, 'b-o', ms=3, lw=1.4)
a1.axhline(100, color='k', lw=0.8, ls='--', alpha=0.6, label='Optimum terbukti')
a1.set_xlabel("Iterasi"); a1.set_ylabel("Hypervolume (% dari optimum terbukti)")
a1.set_title("Konvergensi Hypervolume"); a1.grid(alpha=0.3); a1.legend(fontsize=8)

a2.plot(iters, sizes, 'g-o', ms=3, lw=1.4)
a2.axhline(len(P_star), color='k', lw=0.8, ls='--', alpha=0.6,
           label=f'|P*| = {len(P_star)}')
a2.set_xlabel("Iterasi"); a2.set_ylabel("Ukuran archive")
a2.set_title("Pertumbuhan Archive"); a2.grid(alpha=0.3); a2.legend(fontsize=8)
plt.suptitle(f"Dinamika Konvergensi - Artikel {AID}", y=1.02)
plt.tight_layout()
plt.savefig(f"{REPORT_FIG_DIR}/babiv_A_konvergensi_{AID}.png",
            dpi=200, bbox_inches='tight')
plt.show()

# gambar B: recovery front Pareto
plt.figure(figsize=(7, 4.2))
plt.plot(iters, rec_pct, 'm-o', ms=3, lw=1.4)
plt.axhline(100, color='k', lw=0.8, ls='--', alpha=0.6)
plt.xlabel("Iterasi"); plt.ylabel("Front recovery (%)")
plt.title(f"Pemulihan Front Pareto Sebenarnya - Artikel {AID}")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{REPORT_FIG_DIR}/babiv_B_recovery_{AID}.png",
            dpi=200, bbox_inches='tight')
plt.show()

# gambar C: archive akhir vs P*
plt.figure(figsize=(7.6, 5.4))
plt.scatter(cov_all, red_all, s=4, c='#e8e8e8', zorder=1,
            label=f'{len(combos):,} kandidat feasible')
plot_pareto_curve(np.column_stack([-Ps[:, 0], Ps[:, 1]]), color='k', lw=1.6,
            label=f'Front Pareto sebenarnya (|P*| = {len(Ps)})')
plt.scatter(fin[:, 0], fin[:, 1], s=110, marker='x', c='tab:red', lw=2.2,
            zorder=3, label=f'Archive MO-WOA+ ({len(fin)} solusi)')
plt.scatter([lh[-1]['knee_cov']], [lh[-1]['knee_red']], s=300, marker='*',
            c='gold', edgecolors='k', zorder=4, label='Knee-point')
plt.xlabel("Coverage ↑"); plt.ylabel("Redundancy ↓")
plt.title(f"Archive Akhir vs Front Pareto Sebenarnya - Artikel {AID}")
plt.legend(fontsize=8); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(f"{REPORT_FIG_DIR}/babiv_C_front_{AID}.png",
            dpi=200, bbox_inches='tight')
plt.show()

# gambar D: archive diwarnai iterasi ditemukan
# (overlay snapshot tidak berguna karena archive monoton, titik lama tertutup)
plt.figure(figsize=(7.6, 5.4))
plot_pareto_curve(np.column_stack([-Ps[:, 0], Ps[:, 1]]), color='k', lw=1.2,
            style='--', dashed_only=True,
            label=f'Front Pareto sebenarnya (|P*| = {len(Ps)})')
sc = plt.scatter(fin[:, 0], fin[:, 1], c=disc, cmap='viridis', s=150,
                 edgecolors='k', linewidths=0.7, zorder=3)
plt.scatter([lh[-1]['knee_cov']], [lh[-1]['knee_red']], s=340, marker='*',
            facecolors='none', edgecolors='red', linewidths=2, zorder=4,
            label='Knee-point')
cb = plt.colorbar(sc)
cb.set_label("Iterasi saat solusi pertama ditemukan")
plt.xlabel("Coverage ↑"); plt.ylabel("Redundancy ↓")
plt.title(f"Ekspansi Archive - Artikel {AID}")
plt.legend(fontsize=8, loc='upper left'); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(f"{REPORT_FIG_DIR}/babiv_D_ekspansi_{AID}.png",
            dpi=200, bbox_inches='tight')
plt.show()

# tabel 1
t_last = iters[-1]
tabel1 = [
    ("Jumlah kalimat (N)", N),
    ("Ruang pencarian C(N,3)", f"{len(combos):,}"),
    ("Evaluasi fungsi (budget)", f"1480 ({100*1480/len(combos):.1f}% ruang)"),
    ("|P*| front sebenarnya (enumerasi eksak)", len(P_star)),
    ("|S| archive akhir", len(fin)),
    ("Front recovery akhir", f"{rec_fin}/{len(P_star)} ({100*rec_fin/len(P_star):.1f}%)"),
    ("Kapasitas archive (Tabel 4.8)",
     f"{mr.get('archive_size','-')}   -> pruning "
     f"{'AKTIF' if len(fin) >= mr.get('archive_size',10**9) else 'tidak pernah aktif'}"),
    ("HV iterasi 1", f"{hv_pct[0]:.2f}% dari optimum"),
    (f"HV iterasi {t_last}", f"{hv_pct[-1]:.2f}% dari optimum"),
    ("Kenaikan HV", f"+{hv_pct[-1]-hv_pct[0]:.2f} poin"),
    ("Recovery iterasi 1", f"{rec_pct[0]:.1f}%"),
    (f"Recovery iterasi {t_last}", f"{rec_pct[-1]:.1f}%"),
    ("Kenaikan recovery", f"+{rec_pct[-1]-rec_pct[0]:.1f} poin"),
    (f"Redundansi terendah, iterasi 1 -> {t_last}",
     f"{lh[0]['red_min']:.4f} -> {lh[-1]['red_min']:.4f} "
     f"({100*(lh[0]['red_min']-lh[-1]['red_min'])/max(lh[0]['red_min'],1e-12):.1f}% turun)"),
    (f"Coverage tertinggi, iterasi 1 -> {t_last}",
     f"{lh[0]['cov_max']:.4f} -> {lh[-1]['cov_max']:.4f}"),
    ("Coverage maksimum teoretis", f"{cov_all.max():.4f}"),
    ("Solusi dari inisialisasi (iterasi 1)", f"{(disc==1).sum()}/{len(disc)}"),
    ("Solusi ditemukan pencarian (iterasi >1)", f"{(disc>1).sum()}/{len(disc)}"),
    ("Iterasi penemuan terakhir", disc.max()),
]
print(f"\ntabel 1: ringkasan metrik, artikel {AID}")
for nama, nilai in tabel1:
    print(f"{nama:<44}{nilai}")

# tabel 2
print("\ntabel 2: recovery & HV per iterasi (tab-separated)")
print("Iterasi\tUkuran archive\tHV (%)\tRecovery (%)\tRedundansi min")
marks = sorted(set([0, 4, 9, 11, 19, 24, 34, 37, len(lh)-1]))
for i in marks:
    if i < len(lh):
        print(f"{iters[i]}\t{sizes[i]}\t{hv_pct[i]:.2f}\t{rec_pct[i]:.1f}\t"
              f"{lh[i]['red_min']:.4f}")

# tabel 3
print("\ntabel 3: solusi archive akhir, urut coverage turun (tab-separated)")
print("#\tCoverage\tRedundancy\tDitemukan iterasi\tPada P*?\tTerpilih?")
order = np.argsort(-fin[:, 0])
knee_pt = np.array([lh[-1]['knee_cov'], lh[-1]['knee_red']])
for rank, i in enumerate(order, 1):
    on_ps = np.any(np.all(np.isclose(P_star,
                   np.array([-fin[i, 0], fin[i, 1]])), axis=1))
    is_knee = np.allclose(fin[i], knee_pt)
    print(f"{rank}\t{fin[i,0]:.6f}\t{fin[i,1]:.6f}\t{disc[i]}\t"
          f"{'Ya' if on_ps else 'Tidak'}\t{'KNEE-POINT' if is_knee else ''}")


# gambar E (gambar 4.6): front pareto + lapisan non-dominasi 2..EDGE_LAYERS
# merah = P*, hitam = lapisan 2-5, abu-abu = sisanya
EDGE_LAYERS = 5

def nds_layers_2d(cov, red, max_layer):
    """non-dominated sorting 2D, return nomor lapisan tiap titik (1 = front)"""
    layer = np.full(len(cov), max_layer + 1, dtype=int)
    remaining = np.arange(len(cov))
    for k in range(1, max_layer + 1):
        if len(remaining) == 0:
            break
        c, r = cov[remaining], red[remaining]
        order = np.lexsort((r, -c))          # coverage turun, lalu redundancy naik
        front, best = [], np.inf
        for i in order:
            if r[i] < best - 1e-12:
                front.append(i)
                best = r[i]
        idx = remaining[front]
        layer[idx] = k
        remaining = np.setdiff1d(remaining, idx)
    return layer

layer    = nds_layers_2d(cov_all, red_all, EDGE_LAYERS)
is_front = layer == 1
is_edge  = (layer >= 2) & (layer <= EDGE_LAYERS)
is_rest  = layer > EDGE_LAYERS
assert is_front.sum() == len(Ps), "Lapisan 1 harus sama dengan front Pareto P*"

plt.figure(figsize=(7.8, 5.6))
plt.scatter(cov_all[is_rest], red_all[is_rest], s=70, c='#b9bfca',
            edgecolors='none', zorder=1, label='Ruang solusi feasible')
plt.scatter(cov_all[is_edge], red_all[is_edge], s=130, c='black',
            edgecolors='none', zorder=2)
plt.scatter(cov_all[is_front], red_all[is_front], s=170, c='#d0121b',
            edgecolors='none', zorder=3, label='Solusi non-dominan')

plt.xlabel("Coverage (Maximize) →")
plt.ylabel("Redundancy (Minimize) ↓")
plt.title(f"Front Pareto - Artikel {AID}")
plt.legend(loc='upper right', fontsize=9, framealpha=0.95, markerscale=0.6)
plt.grid(alpha=0.22)
plt.tight_layout()
plt.savefig(f"{REPORT_FIG_DIR}/babiv_E_front_scatter_{AID}.png",
            dpi=200, bbox_inches='tight')
plt.show()

print(f"\nGambar E: {len(combos):,} kandidat feasible = {is_front.sum()} merah (front) "
      f"+ {is_edge.sum()} hitam (lapisan 2-{EDGE_LAYERS}) + {is_rest.sum():,} abu-abu")

In [ ]:
GREY, BLACK = '#c3ccd6', '#2b3a48'
same = (i_knee == i_front)

SPECS = []
if same:
    SPECS.append(('(1,2)', i_knee, '#1f5fa8', '*', 190,
                  f'(1,2) Delivered = best on front, ROUGE-1 = {r1[i_knee]:.3f}'))
else:
    SPECS.append(('(1)', i_knee, '#1f5fa8', '*', 190,
                  f'(1) Delivered (knee-point), ROUGE-1 = {r1[i_knee]:.3f}'))
    SPECS.append(('(2)', i_front, '#c0451b', 'D', 45,
                  f'(2) Best on front, ROUGE-1 = {r1[i_front]:.3f}'))
SPECS.append(('(3)', i_oracle, '#2a8f36', 'X', 70,
              f'(3) Global oracle (dominated), ROUGE-1 = {r1[i_oracle]:.3f}'))

with plt.rc_context({'font.size': 7, 'font.family': 'serif'}):
    fig, ax = plt.subplots(figsize=(3.45, 3.15))
    ax.scatter(cov_all[_dom], red_all[_dom], s=3, c=GREY, alpha=0.6,
               edgecolors='none', zorder=1,
               label=f'Dominated candidates ({_dom.sum():,})')
    ax.scatter(-Ps[:, 0], Ps[:, 1], s=13, c=BLACK, edgecolors='none',
               zorder=3, label=f'True Pareto front (|P*| = {len(Ps)})')

    for tag, i, col, mk, sz, legtext in SPECS:
        ax.scatter([cov_all[i]], [red_all[i]], marker=mk, s=sz, c=col,
                   edgecolors='white', linewidths=0.6, zorder=6, label=legtext)
        ax.annotate(tag, xy=(cov_all[i], red_all[i]),
                    xytext=(0, -11), textcoords='offset points',
                    fontsize=6.5, color=col, fontweight='bold',
                    ha='center', va='top', zorder=7)

    if not _on[i_oracle]:
        oc, orr = cov_all[i_oracle], red_all[i_oracle]
        cand = [(abs(-Ps[j, 0] - oc) + abs(Ps[j, 1] - orr), j)
                for j in range(len(Ps))
                if -Ps[j, 0] >= oc and Ps[j, 1] <= orr]
        if cand:
            j = min(cand)[1]
            ax.plot([oc, -Ps[j, 0]], [orr, Ps[j, 1]], ':',
                    color='#2a8f36', lw=0.8, zorder=5)

    ax.set_xlabel('Coverage (Maximize) \u2192')
    ax.set_ylabel('Redundancy (Minimize) \u2193')
    ax.grid(alpha=0.22, lw=0.5); ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.legend(fontsize=5.6, loc='upper center', bbox_to_anchor=(0.5, -0.20),
              frameon=False, handletextpad=0.4, labelspacing=0.35)
    fig.tight_layout(pad=0.2)
    fig.savefig(f'{REPORT_FIG_DIR}/paper_fig1_{AID}.pdf', bbox_inches='tight')
    fig.savefig(f'{REPORT_FIG_DIR}/paper_fig1_{AID}.png', dpi=300,
                bbox_inches='tight')
    plt.show()

In [ ]:
# gambar 4.7: front pareto dua artikel sulit lain, untuk bandingkan bentuk front
# P* dihitung per artikel (enumerasi C(N,3)), MOWOA+ hanya dijalankan ulang di 2 artikel ini
import io, contextlib
from itertools import combinations
from math import comb
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

NUM_ARTICLES     = 150
K                = 3
FE_BUDGET        = 1480
EDGE_LAYERS      = 5
EXCLUDE_IDX      = {148}      # artikel 149 sudah dipakai di gambar 4.6
FORCE_WIDE_IDX   = None       # isi index 0-based kalau mau pilih manual
FORCE_SPARSE_IDX = None
RUN_MOWOA_CHECK  = True
LEGEND_LOC       = 'upper right'


def position_bonus_for(n):
    pb = np.ones(n)
    pb[0] = 3.0
    pb[1:3] *= 2.5
    pb[3:5] *= 1.5
    pb[5:8] *= 0.8
    pb[8:] *= 0.3
    return pb


def nds_layers_2d(cov, red, max_layer):
    """non-dominated sorting, sama dengan cell sebelumnya"""
    layer = np.full(len(cov), max_layer + 1, dtype=int)
    remaining = np.arange(len(cov))
    for k in range(1, max_layer + 1):
        if len(remaining) == 0:
            break
        c, r = cov[remaining], red[remaining]
        order = np.lexsort((r, -c))
        front, best = [], np.inf
        for i in order:
            if r[i] < best - 1e-12:
                front.append(i)
                best = r[i]
        idx_k = remaining[front]
        layer[idx_k] = k
        remaining = np.setdiff1d(remaining, idx_k)
    return layer


def exact_feasible_set(idx):
    """enumerasi semua ringkasan K kalimat untuk artikel idx"""
    global report_mode
    report_mode = False
    sentences = split_text_into_sentences(test_df.paragraphs[idx])
    with contextlib.redirect_stdout(io.StringIO()):
        graph, _ = run_bge_m3_and_output_graph(sentences, article_id=idx + 1)
        compute_lexrank_importance(graph)
    n = len(sentences)
    pb = position_bonus_for(n)
    bi = np.array([graph.nodes[i]['importance'] * pb[i] for i in range(n)])
    adj = nx.to_numpy_array(graph, weight='weight')
    cmb = np.array(list(combinations(range(n), K)), dtype=int)
    iu, ju = np.triu_indices(K, k=1)
    cov = bi[cmb].sum(axis=1)
    red = np.maximum(0.0, adj[cmb[:, iu], cmb[:, ju]].sum(axis=1) / len(iu) - 0.2)
    layer = nds_layers_2d(cov, red, EDGE_LAYERS)
    front = layer == 1
    cf = np.sort(cov[front])
    span = cf[-1] - cf[0]
    rng_all = cov.max() - cov.min()
    gaps = np.diff(cf)
    top = int(np.argmax(cov))
    return dict(idx=idx, N=n, sentences=sentences, graph=graph, pb=pb,
                combos=cmb, cov=cov, red=red, layer=layer,
                n_front=int(front.sum()),
                span_ratio=float(span / rng_all) if rng_all > 0 else 0.0,
                max_gap_ratio=float(gaps.max() / span) if len(gaps) and span > 0 else 0.0,
                top_is_lead3=bool(tuple(cmb[top]) == (0, 1, 2)),
                n_front_with_last=int(np.sum(cmb[front] == n - 1)))


# 1) artikel sulit, kecuali 149
hard_idx = []
for idx in range(NUM_ARTICLES):
    n = len(split_text_into_sentences(test_df.paragraphs[idx]))
    if n >= K and comb(n, K) > FE_BUDGET and idx not in EXCLUDE_IDX:
        hard_idx.append(idx)
print(f"{len(hard_idx)} artikel sulit (C(N,3) > {FE_BUDGET}), tanpa Artikel 149\n")

# 2) P* per artikel + bentuk front
results = [exact_feasible_set(i) for i in hard_idx]
print(f"{'Artikel':>8}{'N':>5}{'C(N,3)':>9}{'|P*|':>6}{'span%':>8}{'gap%':>7}"
      f"{'top=LEAD3':>10}{'P* w/ last':>11}")
for d in sorted(results, key=lambda d: (-d['n_front'], -d['span_ratio'])):
    print(f"{d['idx']+1:>8}{d['N']:>5}{comb(d['N'], K):>9}{d['n_front']:>6}"
          f"{100*d['span_ratio']:>7.1f}%{100*d['max_gap_ratio']:>6.1f}%"
          f"{'ya' if d['top_is_lead3'] else 'tidak':>10}{d['n_front_with_last']:>11}")

# 3) pilih artikel (atau override)
by_idx = {d['idx']: d for d in results}
wide   = by_idx[FORCE_WIDE_IDX] if FORCE_WIDE_IDX is not None else \
         max(results, key=lambda d: (d['n_front'], d['span_ratio']))
sparse = by_idx[FORCE_SPARSE_IDX] if FORCE_SPARSE_IDX is not None else \
         min(results, key=lambda d: (d['n_front'], d['span_ratio']))
print(f"\n(a) front terlebar : Artikel {wide['idx']+1}  (|P*| = {wide['n_front']})")
print(f"(b) front tersempit: Artikel {sparse['idx']+1}  (|P*| = {sparse['n_front']})")

# 4) cek recovery MOWOA+ (seed 0)
if RUN_MOWOA_CHECK:
    for d in (wide, sparse):
        with contextlib.redirect_stdout(io.StringIO()):
            out = mowoa_plus_algorithm_with_pareto(d['graph'], summary_size=K,
                                                   position_bonus=d['pb'], seed=0)
        arch = np.asarray(out[-1]['final_front'], float)
        f = d['layer'] == 1
        P = np.column_stack([d['cov'][f], d['red'][f]])
        hit = sum(np.any(np.all(np.isclose(arch, p, atol=1e-9), axis=1)) for p in P)
        print(f"Artikel {d['idx']+1}: archive MOWOA+ memulihkan {hit}/{len(P)} solusi P*")

# 5) plot, gaya sama dengan gambar 4.6
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, d, tag in zip(axes, (wide, sparse), ('(a)', '(b)')):
    lay = d['layer']
    f, e, g = lay == 1, (lay >= 2) & (lay <= EDGE_LAYERS), lay > EDGE_LAYERS
    ax.scatter(d['cov'][g], d['red'][g], s=70, c='#b9bfca', edgecolors='none',
               zorder=1, label='Ruang solusi feasible')
    ax.scatter(d['cov'][e], d['red'][e], s=130, c='black', edgecolors='none', zorder=2)
    ax.scatter(d['cov'][f], d['red'][f], s=170, c='#d0121b', edgecolors='none',
               zorder=3, label='Solusi non-dominan')
    ax.set_title(f"{tag} Artikel {d['idx'] + 1}")
    ax.set_xlabel("Coverage (Maximize) →")
    ax.set_ylabel("Redundancy (Minimize) ↓")
    ax.grid(alpha=0.22)
    ax.legend(loc=LEGEND_LOC, fontsize=9, framealpha=0.95, markerscale=0.6)
plt.tight_layout()
plt.savefig(f"{REPORT_FIG_DIR}/babiv_front_comparison.png", dpi=200, bbox_inches='tight')
plt.show()

### Save results summaries to a file to be used one evaluation notebook

In [ ]:
import json

# simpan hasil MO-WOA+ (seed 0) untuk notebook evaluasi
mowoa_output = {
    'generated_summaries': generated_summaries,
    'golden_summaries': golden_summaries,
    'source_texts': source_texts,
}

with open('mowoa_plus_summaries_150_seed0.json', 'w', encoding='utf-8') as f:
    json.dump(mowoa_output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(generated_summaries)} MO-WOA+ summaries to mowoa_plus_summaries_150_seed0.json")

In [ ]:
lead3_rouge1 = {'precision': [], 'recall': [], 'f1': []}
lead3_rouge2 = {'precision': [], 'recall': [], 'f1': []}
lead3_rougel = {'precision': [], 'recall': [], 'f1': []}
lead3_summaries = []

NUM_ARTICLES = len(generated_summaries)  # sama dengan jumlah artikel MO-WOA

for idx in range(NUM_ARTICLES):
    current_paragraphs = test_df.paragraphs[idx]
    current_golden_summary = tokens_to_narrative(test_df.summary[idx])

    lead_scores, lead_summary = lead_n_baseline(current_paragraphs, current_golden_summary, n=3)

    for metric_key, score_dict in [('rouge1', lead3_rouge1),
                                    ('rouge2', lead3_rouge2),
                                    ('rougeL', lead3_rougel)]:
        score_dict['precision'].append(lead_scores[metric_key].precision)
        score_dict['recall'].append(lead_scores[metric_key].recall)
        score_dict['f1'].append(lead_scores[metric_key].fmeasure)

    lead3_summaries.append(lead_summary)

print(f"LEAD-3 baseline evaluated on {NUM_ARTICLES} articles.")

In [ ]:
print("MO-WOA vs LEAD-3 (tab-separated)")
print("\t".join(["Metric", "MO-WOA Precision", "MO-WOA Recall", "MO-WOA F1",
                 "LEAD-3 Precision", "LEAD-3 Recall", "LEAD-3 F1", "Δ F1"]))

for name, mowoa_sd, lead_sd in [
    ("ROUGE-1", rouge1_scores, lead3_rouge1),
    ("ROUGE-2", rouge2_scores, lead3_rouge2),
    ("ROUGE-L", rougel_scores, lead3_rougel),
]:
    m_p = np.mean(mowoa_sd['precision'])
    m_r = np.mean(mowoa_sd['recall'])
    m_f = np.mean(mowoa_sd['f1'])
    l_p = np.mean(lead_sd['precision'])
    l_r = np.mean(lead_sd['recall'])
    l_f = np.mean(lead_sd['f1'])
    delta = m_f - l_f
    sign = "+" if delta >= 0 else ""
    print(f"{name}\t{m_p:.4f}\t{m_r:.4f}\t{m_f:.4f}\t"
          f"{l_p:.4f}\t{l_r:.4f}\t{l_f:.4f}\t{sign}{delta:.4f}")


# compression ratio
lead3_source_texts = []
for idx in range(NUM_ARTICLES):
    sentences = split_text_into_sentences(train_df.paragraphs[idx])
    lead3_source_texts.append(' '.join(sentences))

lead3_cr = np.mean([len(s) / max(1, len(src))
                     for s, src in zip(lead3_summaries, lead3_source_texts)])
mowoa_cr = np.mean([len(s) / max(1, len(src))
                     for s, src in zip(generated_summaries, source_texts)])

print(f"\nCR: MO-WOA {mowoa_cr:.1%}, LEAD-3 {lead3_cr:.1%}")
print(f"panjang rata2: MO-WOA {np.mean([len(s) for s in generated_summaries]):.0f}, "
      f"LEAD-3 {np.mean([len(s) for s in lead3_summaries]):.0f} karakter")

# redundansi trigram
print("\nLEAD-3:")
lead3_ngram = compute_ngram_redundancy_batch(lead3_summaries)

# simpan tsv
import os
os.makedirs("report_figures", exist_ok=True)
with open("report_figures/mowoa_vs_lead3.tsv", "w") as f:
    f.write("Metric\tMO-WOA Precision\tMO-WOA Recall\tMO-WOA F1\tLEAD-3 Precision\tLEAD-3 Recall\tLEAD-3 F1\tDelta F1\n")
    for name, mowoa_sd, lead_sd in [
        ("ROUGE-1", rouge1_scores, lead3_rouge1),
        ("ROUGE-2", rouge2_scores, lead3_rouge2),
        ("ROUGE-L", rougel_scores, lead3_rougel),
    ]:
        m_f = np.mean(mowoa_sd['f1'])
        l_f = np.mean(lead_sd['f1'])
        f.write(f"{name}\t{np.mean(mowoa_sd['precision']):.4f}\t{np.mean(mowoa_sd['recall']):.4f}\t{m_f:.4f}\t"
                f"{np.mean(lead_sd['precision']):.4f}\t{np.mean(lead_sd['recall']):.4f}\t{l_f:.4f}\t{m_f - l_f:+.4f}\n")
print("\nsaved: report_figures/mowoa_vs_lead3.tsv")

In [ ]:
# rouge rata-rata semua artikel
print("ROUGE rata-rata (tab-separated)")
print("\t".join(["Metric", "Avg Precision", "Avg Recall", "Avg F1"]))
for name, sd in [("ROUGE-1", rouge1_scores),
                  ("ROUGE-2", rouge2_scores),
                  ("ROUGE-L", rougel_scores)]:
    print(f"{name}\t{np.mean(sd['precision']):.4f}\t{np.mean(sd['recall']):.4f}\t{np.mean(sd['f1']):.4f}")

# bertscore (dimatikan, error overflow di tokenizer indobert)
# print("\n[E2] BERTScore")

# # model_max_length bawaan indobert terlalu besar, set ke 512
# from transformers import AutoTokenizer
# _tok = AutoTokenizer.from_pretrained('indobenchmark/indobert-large-p2')
# _tok.model_max_length = 512
# _tok.save_pretrained('/tmp/indobert-patched')

# valid_gen = []
# valid_ref = []
# skipped = 0
# for g, r in zip(generated_summaries, golden_summaries):
#     if not g.strip() or not r.strip():
#         skipped += 1
#         continue
#     valid_gen.append(' '.join(g.split()[:400]))
#     valid_ref.append(' '.join(r.split()[:400]))

# if skipped > 0:
#     print(f"  Skipped {skipped} articles with empty summaries")

# print(f"  Evaluating {len(valid_gen)} valid pairs")

# # pakai tokenizer lokal yang sudah dipatch
# bert_results = compute_bertscore_metrics(
#     valid_gen, valid_ref,
#     model_type='/tmp/indobert-patched'
# )

# n-gram redundancy
ngram_results = compute_ngram_redundancy_batch(generated_summaries)

# compression ratio
cr_results = compute_compression_ratio(source_texts, generated_summaries)

# tabel gabungan
print("\nrekap (tab-separated)")
print("\t".join(["Metric", "Value"]))
print(f"ROUGE-1 F1\t{np.mean(rouge1_scores['f1']):.4f}")
print(f"ROUGE-2 F1\t{np.mean(rouge2_scores['f1']):.4f}")
print(f"ROUGE-L F1\t{np.mean(rougel_scores['f1']):.4f}")
# print(f"BERTScore F1\t{bert_results['avg_f1']:.4f}")
print(f"Compression Ratio\t{cr_results['avg_compression_ratio']:.4f}")
print(f"N-gram Redundancy\t{ngram_results['avg_redundancy']:.4f}")

# simpan tsv
import os
os.makedirs("report_figures", exist_ok=True)
with open("report_figures/consolidated_evaluation.tsv", "w") as f:
    f.write("Metric\tAvg Precision\tAvg Recall\tAvg F1\n")
    for name, sd in [("ROUGE-1", rouge1_scores),
                      ("ROUGE-2", rouge2_scores),
                      ("ROUGE-L", rougel_scores)]:
        f.write(f"{name}\t{np.mean(sd['precision']):.4f}\t{np.mean(sd['recall']):.4f}\t{np.mean(sd['f1']):.4f}\n")
    # f.write(f"BERTScore\t{bert_results['avg_precision']:.4f}\t{bert_results['avg_recall']:.4f}\t{bert_results['avg_f1']:.4f}\n")
    f.write(f"\nMetric\tValue\n")
    f.write(f"Compression Ratio\t{cr_results['avg_compression_ratio']:.4f}\n")
    f.write(f"N-gram Redundancy\t{ngram_results['avg_redundancy']:.4f}\n")
print("\nsaved: report_figures/consolidated_evaluation.tsv")